# SketchForge 3D
### A sketch → an object image → a downloadable 3D asset

**Complete implementation for a GPU Google Colab notebook.** It accepts an uploaded or drawn sketch and an optional description, runs Stable Diffusion 1.5 with ControlNet Scribble, isolates the object, reconstructs it with TripoSR, and exports a validated GLB. UV texture baking and OBJ export are optional.

**Start:** select **Runtime → Change runtime type → GPU**, then run the cells in order or choose **Run all**. Setup can take several minutes and download several GB. Press **Generate 3D model** in the interface when ready. Start with Preview quality.

**What Run all does:** writes the embedded project, prepares an isolated environment, checks imports and the GPU, runs CPU checks, and opens notebook controls. It does not automatically generate models, run a dataset batch, mount Drive, or launch a public web app.

This notebook contains the complete source. Pretrained weights download from their public repositories at first use. No paid inference API is required.

**Verification:** preprocessing, geometry/export checks, CPU texture baking and failure reporting have local tests. End-to-end neural inference, resource use and the Colab UI must be confirmed in your GPU session. No generated results or speed benchmarks are prefilled.

## 1. Create the project folder

All runtime files live under `/content/sketchforge3d`. Colab can discard this folder when it replaces the runtime; download your results or explicitly save them to Drive.

In [2]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, zipfile
ROOT = Path('/content/sketchforge3d')
ROOT.mkdir(parents=True, exist_ok=True)
print('Project folder:', ROOT)


Project folder: /content/sketchforge3d


## 2. Write the full implementation

The following cells contain the actual pipeline and interface source, not placeholders. You can expand them to inspect or edit the implementation. They write project files without starting inference.

In [3]:
# Embedded project file: engine.py
FILE_CONTENT = '''"""SketchForge: isolated sketch-to-image and TripoSR workers, plus a CLI.

Original integration code. Upstream models retain their respective licenses.
This module does not import torch until an inference worker is started.
"""
from __future__ import annotations

import argparse
import contextlib
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import platform
import subprocess
import sys
import time
import traceback
import types
import uuid
import zipfile

import cv2
import numpy as np
from PIL import Image, ImageOps
import trimesh

VERSION = "1.0.0"
PINS = {
    "triposr_code": "107cefdc244c39106fa830359024f6a2f1c78871",
    "base": {"id": "stable-diffusion-v1-5/stable-diffusion-v1-5", "revision": "451f4fe16113bff5a5d2269ed5ad43b0592e9a14"},
    "controlnet": {"id": "lllyasviel/control_v11p_sd15_scribble", "revision": "3564ec7b87dd706d5fc760d21c9e90bb160d1e9f"},
    "triposr": {"id": "stabilityai/TripoSR", "revision": "5b521936b01fbe1890f6f9baed0254ab6351c04a"},
    "dino_config": {"id": "facebook/dino-vitb16", "revision": "f205d5d8e640a89a2b8ef0369670dfc37cc07fc2"},
}
PRESETS = {
    "preview": {"size": 512, "steps": 20, "mesh_resolution": 128, "texture_size": 512, "chunk_size": 4096},
    "balanced": {"size": 512, "steps": 25, "mesh_resolution": 192, "texture_size": 1024, "chunk_size": 4096},
    "detailed": {"size": 512, "steps": 30, "mesh_resolution": 256, "texture_size": 1024, "chunk_size": 4096},
}


def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False, default=str), encoding="utf-8")
    temporary.replace(path)


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha256(path):
    result = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(block)
    return result.hexdigest()


def read_rgb(source):
    with Image.open(source) as opened:
        if opened.width * opened.height > 25_000_000:
            raise ValueError("Please resize the input to fewer than 25 million pixels.")
        image = ImageOps.exif_transpose(opened).convert("RGBA")
    white = Image.new("RGBA", image.size, "white")
    white.alpha_composite(image)
    return white.convert("RGB")


def square_frame(array, box, margin=0.12, fill=255):
    x0, y0, x1, y1 = box
    side = max(8, int(math.ceil(max(x1 - x0, y1 - y0) * (1 + 2 * margin))))
    left = int(math.floor((x0 + x1 - side) / 2))
    top = int(math.floor((y0 + y1 - side) / 2))
    framed = np.full((side, side) + array.shape[2:], fill, dtype=array.dtype)
    xa, ya = max(left, 0), max(top, 0)
    xb, yb = min(left + side, array.shape[1]), min(top + side, array.shape[0])
    if xa < xb and ya < yb:
        framed[ya - top:yb - top, xa - left:xb - left] = array[ya:yb, xa:xb]
    return framed


def bounding_box(mask):
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        raise ValueError("The input contains no visible sketch strokes.")
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def condition_sketch(source, directory, size=512):
    directory = Path(directory)
    image = read_rgb(source)
    image.save(directory / "00_input.png")
    array = np.asarray(image)
    gray = cv2.cvtColor(array, cv2.COLOR_RGB2GRAY)
    # Inverted sketches are accepted explicitly by inspecting the background.
    border = np.concatenate((gray[0], gray[-1], gray[:, 0], gray[:, -1]))
    inverted = bool(np.median(border) < 128)
    if inverted:
        gray = 255 - gray
        array = 255 - array
    if float(gray.std()) < 2.0:
        raise ValueError("The input is blank. Draw or upload dark strokes on a plain background.")
    threshold, _ = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    threshold = float(np.clip(threshold, 24, 224))
    mask = gray <= threshold
    if mask.sum() < 8 or mask.mean() > 0.65:
        raise ValueError("The sketch is empty or the background is too dark. Use a clear line drawing.")
    box = bounding_box(mask)
    framed_mask = square_frame(mask.astype(np.uint8) * 255, box, fill=0)
    control = cv2.resize(framed_mask, (size, size), interpolation=cv2.INTER_AREA)
    control = np.where(control > 60, 255, 0).astype(np.uint8)
    control = cv2.dilate(control, np.ones((2, 2), np.uint8), iterations=1)
    source_frame = square_frame(array, box)
    Image.fromarray(source_frame).resize((size, size), Image.Resampling.LANCZOS).save(directory / "01_framed_sketch.png")
    Image.fromarray(control).convert("RGB").save(directory / "02_control.png")
    return {"inverted_input": inverted, "stroke_fraction": float((control > 0).mean()), "crop_box": list(box)}


def image_metrics(image, control):
    structure = np.asarray(control.convert("L")) > 0
    gray = cv2.cvtColor(np.asarray(image.convert("RGB")), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (3, 3), 0), 60, 160) > 0
    kernel = np.ones((9, 9), np.uint8)
    near_structure = cv2.dilate(structure.astype(np.uint8), kernel) > 0
    near_edges = cv2.dilate(edges.astype(np.uint8), kernel) > 0
    recall = float((structure & near_edges).sum() / max(int(structure.sum()), 1))
    precision = float((edges & near_structure).sum() / max(int(edges.sum()), 1))
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    a = np.asarray(image.convert("RGB"))
    border = np.concatenate((a[:20].reshape(-1, 3), a[-20:].reshape(-1, 3), a[:, :20].reshape(-1, 3), a[:, -20:].reshape(-1, 3)))
    white = float((border.mean(axis=1) > 220).mean())
    return {"edge_f1_heuristic": round(f1, 5), "white_border_fraction": round(white, 5), "ranking_score": round(0.85 * f1 + 0.15 * white, 5)}


def prepare_foreground(product_path, directory, foreground_ratio=0.85):
    from rembg import new_session, remove
    image = read_rgb(product_path)
    session = new_session("isnet-general-use", providers=["CPUExecutionProvider"])
    rgba = np.asarray(remove(image, session=session).convert("RGBA"))
    alpha = rgba[..., 3]
    coverage = float((alpha > 127).mean())
    if not 0.01 < coverage < 0.99:
        raise ValueError("Background removal did not isolate an object. Inspect the generated image and try a clearer sketch or description.")
    # Retain small components and soft alpha: they may be handles or legs.
    margin = (1 / foreground_ratio - 1) / 2
    framed = square_frame(rgba, bounding_box(alpha > 127), margin=margin, fill=0)
    rgba_image = Image.fromarray(framed, "RGBA").resize((512, 512), Image.Resampling.LANCZOS)
    directory = Path(directory)
    rgba_image.save(directory / "04_foreground.png")
    rgba_image.getchannel("A").save(directory / "04_mask.png")
    f = np.asarray(rgba_image).astype(np.float32) / 255
    composite = f[..., :3] * f[..., 3:4] + 0.5 * (1 - f[..., 3:4])
    Image.fromarray(np.round(composite * 255).astype(np.uint8)).save(directory / "05_triposr_input.png")
    return {"method": "rembg/isnet-general-use", "provider": "CPUExecutionProvider", "original_coverage": coverage, "foreground_ratio": foreground_ratio, "small_components_removed": False}


def require_gpu():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No NVIDIA GPU is available. In Colab select Runtime > Change runtime type > GPU, then rerun setup.")
    torch.cuda.reset_peak_memory_stats()
    props = torch.cuda.get_device_properties(0)
    return {"name": props.name, "total_gib": round(props.total_memory / 1024 ** 3, 2), "torch": torch.__version__, "cuda_build": torch.version.cuda}


def gpu_usage():
    import torch
    torch.cuda.synchronize()
    return {"peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 1024 ** 3, 3), "peak_reserved_gib": round(torch.cuda.max_memory_reserved() / 1024 ** 3, 3)}


def generate_worker(directory):
    import torch
    from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler
    directory = Path(directory)
    cfg = load_json(directory / "config.json")
    t0 = time.perf_counter()
    hardware = require_gpu()
    print("Loading image generation models; the first run downloads weights.", flush=True)
    controlnet = ControlNetModel.from_pretrained(**{"pretrained_model_name_or_path": PINS["controlnet"]["id"], "revision": PINS["controlnet"]["revision"], "torch_dtype": torch.float16, "variant": "fp16", "use_safetensors": True})
    pipe = StableDiffusionControlNetPipeline.from_pretrained(PINS["base"]["id"], revision=PINS["base"]["revision"], controlnet=controlnet, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    # Keep the model's default safety checker enabled.
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_slicing()
    control = Image.open(directory / "02_control.png").convert("RGB")
    subject = cfg["prompt"].strip() or "a single solid physical object matching the supplied sketch"
    prompt = f"studio product photograph of {subject}, preserve the outline and proportions of the supplied drawing, same viewpoint, one complete object, plain white background, soft even lighting, realistic surfaces"
    negative = "multiple objects, duplicate object, room, scenery, cropped object, text, watermark, blurry, out of frame"
    candidates = []
    for number in range(cfg["candidates"]):
        seed = (cfg["seed"] + number) % (2 ** 31)
        print(f"Generating candidate {number + 1}/{cfg['candidates']} (seed {seed}).", flush=True)
        with torch.inference_mode():
            result = pipe(prompt=prompt, negative_prompt=negative, image=control, height=cfg["size"], width=cfg["size"], num_inference_steps=cfg["steps"], guidance_scale=7.5, controlnet_conditioning_scale=cfg["control_strength"], generator=torch.Generator(device="cpu").manual_seed(seed), num_images_per_prompt=1)
        flags = result.nsfw_content_detected
        if flags is not None and bool(flags[0]):
            print("This candidate was filtered by the image model; it will not be reconstructed.", flush=True)
            continue
        image = result.images[0]
        name = f"03_candidate_{number + 1}.png"
        image.save(directory / name)
        candidates.append({"file": name, "seed": seed, **image_metrics(image, control)})
    if not candidates:
        raise RuntimeError("No usable image was generated. Try a clearer object sketch or a different description.")
    candidates.sort(key=lambda item: item["ranking_score"], reverse=True)
    with Image.open(directory / candidates[0]["file"]) as chosen:
        chosen.save(directory / "03_product.png")
    save_json(directory / "generation.json", {"hardware": hardware, "memory": gpu_usage(), "duration_s": round(time.perf_counter() - t0, 3), "full_prompt": prompt, "models": {key: PINS[key] for key in ("base", "controlnet")}, "selected_seed": candidates[0]["seed"], "candidates": candidates, "note": "Edge agreement is a 2D ranking heuristic, not a measure of 3D correctness."})


def marching_cubes_numpy(volume, threshold):
    """Match torchmcubes axis ordering; TripoSR itself swaps it back."""
    import mcubes
    vertices, faces = mcubes.marching_cubes(np.asarray(volume, dtype=np.float32), float(threshold))
    return np.ascontiguousarray(vertices[:, ::-1], dtype=np.float32), np.ascontiguousarray(faces, dtype=np.int64)


def install_cpu_marching_cubes():
    import torch
    module = types.ModuleType("torchmcubes")

    def marching_cubes(volume, threshold):
        vertices, faces = marching_cubes_numpy(volume.detach().float().cpu().numpy(), threshold)
        return torch.from_numpy(vertices), torch.from_numpy(faces)

    module.marching_cubes = marching_cubes
    sys.modules["torchmcubes"] = module


def clean_mesh(mesh, target_faces=0):
    if len(mesh.vertices) == 0 or len(mesh.faces) == 0:
        raise ValueError("TripoSR returned empty geometry.")
    if not np.isfinite(mesh.vertices).all():
        raise ValueError("TripoSR returned invalid vertex coordinates.")
    mesh = mesh.copy()
    mesh.update_faces(mesh.nondegenerate_faces())
    mesh.update_faces(mesh.unique_faces())
    mesh.remove_unreferenced_vertices()
    # Deliberately retain disconnected components; geometry is not repaired by deleting them.
    if target_faces and len(mesh.faces) > target_faces:
        original = mesh
        mesh = mesh.simplify_quadric_decimation(face_count=int(target_faces))
        if original.visual.kind == "vertex":
            from scipy.spatial import cKDTree
            nearest = cKDTree(original.vertices).query(mesh.vertices)[1]
            mesh.visual = trimesh.visual.ColorVisuals(mesh, vertex_colors=original.visual.vertex_colors[nearest])
    trimesh.repair.fix_normals(mesh, multibody=True)
    return mesh


def geometry_report(mesh):
    nv, nf = len(mesh.vertices), len(mesh.faces)
    finite = bool(nv and np.isfinite(mesh.vertices).all())
    extents = np.asarray(mesh.extents if finite else [0, 0, 0], dtype=float)
    depth_present = bool(extents.max() > 0 and extents.min() / extents.max() > 0.005)
    checks = {"vertices_present": nv > 0, "faces_present": nf > 0, "finite_vertices": finite, "depth_present": depth_present, "positive_area": bool(nf and np.isfinite(mesh.area) and mesh.area > 0)}
    return {"passed": all(checks.values()), "checks": checks, "vertices": nv, "faces": nf, "extents": extents.tolist(), "watertight": bool(nf and mesh.is_watertight), "winding_consistent": bool(nf and mesh.is_winding_consistent), "components": len(mesh.split(only_watertight=False)) if nf else 0, "note": "Geometry checks detect invalid files and flat output; visual fidelity must also be inspected."}


def rasterize_uv_positions(vertices, faces, uv, resolution):
    """Rasterize positions into a PIL-oriented UV atlas without OpenGL/EGL."""
    positions = np.zeros((resolution, resolution, 3), dtype=np.float32)
    covered = np.zeros((resolution, resolution), dtype=bool)
    pixels = np.column_stack((uv[:, 0] * resolution, (1 - uv[:, 1]) * resolution))
    for face in faces:
        tri = pixels[face]
        lower = np.maximum(np.floor(tri.min(axis=0)).astype(int), 0)
        upper = np.minimum(np.ceil(tri.max(axis=0)).astype(int), resolution - 1)
        if (upper < lower).any():
            continue
        xx, yy = np.meshgrid(np.arange(lower[0], upper[0] + 1) + 0.5, np.arange(lower[1], upper[1] + 1) + 0.5)
        a, b, c = tri
        denom = (b[1] - c[1]) * (a[0] - c[0]) + (c[0] - b[0]) * (a[1] - c[1])
        if abs(denom) < 1e-10:
            continue
        w0 = ((b[1] - c[1]) * (xx - c[0]) + (c[0] - b[0]) * (yy - c[1])) / denom
        w1 = ((c[1] - a[1]) * (xx - c[0]) + (a[0] - c[0]) * (yy - c[1])) / denom
        w2 = 1 - w0 - w1
        inside = (w0 >= -1e-6) & (w1 >= -1e-6) & (w2 >= -1e-6)
        ys, xs = np.nonzero(inside)
        if not len(xs):
            continue
        weights = np.column_stack((w0[inside], w1[inside], w2[inside]))
        target_y, target_x = ys + lower[1], xs + lower[0]
        positions[target_y, target_x] = weights @ vertices[face]
        covered[target_y, target_x] = True
    if not covered.any():
        raise ValueError("UV unwrapping produced no rasterizable triangles.")
    return positions, covered


def bake_texture_cpu_atlas(mesh, resolution, color_query):
    import xatlas
    from scipy.ndimage import distance_transform_edt
    atlas = xatlas.Atlas()
    atlas.add_mesh(np.asarray(mesh.vertices, dtype=np.float32), np.asarray(mesh.faces, dtype=np.uint32))
    packing = xatlas.PackOptions()
    packing.resolution = int(resolution)
    packing.padding = 4
    atlas.generate(pack_options=packing)
    mapping, indices, uv = atlas[0]
    verts = np.asarray(mesh.vertices)[mapping]
    positions, covered = rasterize_uv_positions(verts, indices, uv, resolution)
    colors = np.asarray(color_query(positions[covered]), dtype=np.float32)
    if colors.shape != (int(covered.sum()), 3) or not np.isfinite(colors).all():
        raise ValueError("Texture color queries returned invalid data.")
    texture = np.zeros((resolution, resolution, 3), dtype=np.uint8)
    texture[covered] = np.round(np.clip(colors, 0, 1) * 255).astype(np.uint8)
    nearest = distance_transform_edt(~covered, return_distances=False, return_indices=True)
    texture[~covered] = texture[nearest[0][~covered], nearest[1][~covered]]
    texture_image = Image.fromarray(texture, "RGB")
    material = trimesh.visual.material.PBRMaterial(baseColorTexture=texture_image, baseColorFactor=[1, 1, 1, 1], metallicFactor=0.0, roughnessFactor=0.75)
    output = trimesh.Trimesh(vertices=verts, faces=indices, process=False)
    output.visual = trimesh.visual.TextureVisuals(uv=uv, material=material)
    return output, texture_image


def normalize_mesh(mesh):
    # Same two rotations used by the upstream TripoSR viewer utility.
    mesh.apply_transform(trimesh.transformations.rotation_matrix(-math.pi / 2, [1, 0, 0]))
    mesh.apply_transform(trimesh.transformations.rotation_matrix(math.pi / 2, [0, 1, 0]))
    mesh.apply_translation(-mesh.bounds.mean(axis=0))
    mesh.apply_scale(1 / max(float(mesh.extents.max()), 1e-9))
    return mesh


def inspect_glb(path, expect_texture=False, expect_vertex_colors=False):
    scene = trimesh.load(path, force="scene", process=False)
    meshes = [g for g in scene.geometry.values() if isinstance(g, trimesh.Trimesh)]
    if not meshes:
        raise ValueError("The exported GLB contains no triangle mesh.")
    reports = [geometry_report(mesh) for mesh in meshes]
    has_texture = False
    has_vertex_colors = False
    for mesh in meshes:
        if mesh.visual.kind == "texture":
            material = mesh.visual.material
            texture = getattr(material, "baseColorTexture", None)
            if texture is None:
                texture = getattr(material, "image", None)
            has_texture = has_texture or bool(texture is not None and getattr(mesh.visual, "uv", None) is not None)
        if mesh.visual.kind == "vertex":
            has_vertex_colors = has_vertex_colors or len(mesh.visual.vertex_colors) == len(mesh.vertices)
    appearance_ok = (has_texture if expect_texture else True) and (has_vertex_colors if expect_vertex_colors else True)
    result = {"passed": all(r["passed"] for r in reports) and appearance_ok, "meshes": reports, "has_texture": has_texture, "has_vertex_colors": has_vertex_colors, "appearance_preserved": appearance_ok, "bytes": Path(path).stat().st_size, "sha256": sha256(path)}
    return result


def write_obj(mesh, directory):
    from trimesh.exchange.obj import export_obj
    # SimpleMaterial gives OBJ an MTL + PNG rather than assuming PBR support.
    obj_mesh = mesh.copy()
    if obj_mesh.visual.kind == "texture":
        mat = obj_mesh.visual.material
        texture = getattr(mat, "baseColorTexture", None)
        if texture is None:
            texture = getattr(mat, "image", None)
        obj_mesh.visual.material = trimesh.visual.material.SimpleMaterial(image=texture)
    text, files = export_obj(obj_mesh, return_texture=True, include_texture=True, mtl_name="model.mtl")
    (Path(directory) / "model.obj").write_text(text, encoding="utf-8")
    for filename, content in (files or {}).items():
        if Path(filename).name != filename:
            raise ValueError("Unexpected relative export path.")
        (Path(directory) / filename).write_bytes(content)


def reconstruct_worker(directory):
    import torch
    from huggingface_hub import hf_hub_download, snapshot_download
    directory = Path(directory)
    cfg = load_json(directory / "config.json")
    hardware = require_gpu()
    t0 = time.perf_counter()
    install_cpu_marching_cubes()
    sys.path.insert(0, cfg["triposr_dir"])
    import tsr.models.tokenizers.image as dino_module
    from tsr.system import TSR

    def download_dino(**kwargs):
        if kwargs.get("repo_id") == PINS["dino_config"]["id"]:
            kwargs["revision"] = PINS["dino_config"]["revision"]
        return hf_hub_download(**kwargs)

    dino_module.hf_hub_download = download_dino
    weights = snapshot_download(PINS["triposr"]["id"], revision=PINS["triposr"]["revision"], allow_patterns=["config.yaml", "model.ckpt"])
    print("Loading TripoSR (float32); no substitute reconstruction backend is used.", flush=True)
    model = TSR.from_pretrained(weights, config_name="config.yaml", weight_name="model.ckpt").to("cuda").eval()
    model.renderer.set_chunk_size(cfg["chunk_size"])
    with torch.inference_mode():
        scene_codes = model([Image.open(directory / "05_triposr_input.png").convert("RGB")], device="cuda")
        mesh = model.extract_mesh(scene_codes, True, resolution=cfg["mesh_resolution"])[0]
    raw_report = geometry_report(mesh)
    mesh = clean_mesh(mesh, target_faces=cfg["target_faces"])
    if not geometry_report(mesh)["passed"]:
        raise ValueError("The reconstructed geometry failed validation. Inspect the foreground image and try another input.")
    warnings = []
    appearance = "vertex_colors"
    if cfg["bake_texture"]:
        print(f"Baking a {cfg['texture_size']}px texture with CPU UV rasterization.", flush=True)

        def query(points):
            colors = []
            with torch.inference_mode():
                for offset in range(0, len(points), cfg["chunk_size"]):
                    batch = torch.from_numpy(points[offset:offset + cfg["chunk_size"]].astype(np.float32)).to("cuda")
                    colors.append(model.renderer.query_triplane(model.decoder, batch, scene_codes[0])["color"].detach().cpu().numpy())
            return np.concatenate(colors)

        try:
            textured, texture = bake_texture_cpu_atlas(mesh, cfg["texture_size"], query)
            mesh = textured
            texture.save(directory / "texture.png")
            appearance = "uv_texture"
        except torch.cuda.OutOfMemoryError:
            raise
        except Exception as error:
            if cfg["require_texture"]:
                raise RuntimeError(f"Required texture baking failed: {error}") from error
            warnings.append(f"Texture baking failed; the exported model uses vertex colors: {type(error).__name__}: {error}")
            print(warnings[-1], flush=True)
    final_mesh = normalize_mesh(mesh)
    if not geometry_report(final_mesh)["passed"]:
        raise ValueError("The final mesh failed geometry validation.")
    glb = directory / "model.glb"
    final_mesh.export(glb)
    check = inspect_glb(glb, expect_texture=appearance == "uv_texture", expect_vertex_colors=appearance == "vertex_colors")
    if not check["passed"]:
        raise ValueError(f"Export validation failed: {check}")
    obj_status = "not_requested"
    if cfg["export_obj"]:
        try:
            write_obj(final_mesh, directory)
            obj_status = "exported; GLB is the authoritative validated output"
        except Exception as error:
            warnings.append(f"Optional OBJ export failed: {error}")
            obj_status = "failed"
    save_json(directory / "reconstruction.json", {"backend": "TripoSR", "hardware": hardware, "memory": gpu_usage(), "duration_s": round(time.perf_counter() - t0, 3), "raw_geometry": raw_report, "final_geometry": geometry_report(final_mesh), "export_validation": check, "appearance": appearance, "obj": obj_status, "warnings": warnings, "models": {"triposr": PINS["triposr"], "dino_config": PINS["dino_config"], "code_commit": PINS["triposr_code"]}, "adapters": ["CPU PyMCubes adapter with TripoSR axis convention", "DINO config revision pin", "CPU UV rasterization with GPU color queries"]})


def package_versions():
    names = ["torch", "torchvision", "diffusers", "transformers", "accelerate", "huggingface-hub", "numpy", "Pillow", "opencv-python-headless", "rembg", "onnxruntime", "trimesh", "PyMCubes", "xatlas", "scipy", "fast-simplification"]
    versions = {}
    for name in names:
        with contextlib.suppress(importlib.metadata.PackageNotFoundError):
            versions[name] = importlib.metadata.version(name)
    return {"python": sys.version, "platform": platform.platform(), "packages": versions}


def worker(stage, directory):
    command = [sys.executable, str(Path(__file__).resolve()), stage, "--job", str(directory)]
    with open(Path(directory) / f"{stage}.log", "w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            with contextlib.suppress(subprocess.TimeoutExpired):
                process.wait(timeout=10)
            if process.poll() is None:
                process.kill()
                process.wait()
            raise
    if code:
        tail = (Path(directory) / f"{stage}.log").read_text(encoding="utf-8")[-1800:]
        raise RuntimeError(f"The {stage} stage failed. See {stage}.log.\\n{tail}")


def bundle_run(directory):
    directory = Path(directory)
    destination = directory / "result_bundle.zip"
    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
        for file in sorted(directory.iterdir()):
            if file.is_file() and file != destination:
                archive.write(file, arcname=file.name)
    return destination


def run_pipeline(args):
    directory = Path(args.run_dir).resolve()
    directory.mkdir(parents=True, exist_ok=False)
    cfg = {**PRESETS[args.quality], "prompt": args.prompt, "seed": int(args.seed) if args.seed >= 0 else int.from_bytes(os.urandom(4), "little") % (2 ** 31), "quality": args.quality, "candidates": args.candidates, "control_strength": args.control_strength, "bake_texture": args.texture, "require_texture": args.require_texture, "target_faces": args.target_faces, "export_obj": args.obj, "triposr_dir": str(Path(args.triposr_dir).resolve())}
    save_json(directory / "config.json", cfg)
    report = {"version": VERSION, "status": "running", "stage": "conditioning", "model_pins": PINS, "config": cfg, "environment": package_versions(), "input_sha256": None, "timings_s": {}, "limitations": ["Single-view hidden surfaces are inferred.", "An empty text prompt is accepted, but ambiguous sketches can be misinterpreted.", "Mesh checks do not establish semantic or manufacturing accuracy."]}
    start = time.perf_counter()
    report_path = directory / "report.json"

    def stage(name, action):
        report["stage"] = name
        save_json(report_path, report)
        print(f"\\n--- {name} ---", flush=True)
        t0 = time.perf_counter()
        result = action()
        report["timings_s"][name] = round(time.perf_counter() - t0, 3)
        return result

    try:
        report["input_sha256"] = sha256(args.input)
        if args.product_image:
            # Explicit reconstruction-only diagnostic, never substituted silently.
            image = read_rgb(args.input)
            image.save(directory / "00_input.png")
            image.save(directory / "03_product.png")
            report["conditioning"] = {"mode": "product_image_diagnostic", "sketch_conditioning_skipped": True}
            report["generation"] = {"explicitly_skipped": True, "reason": "User selected product-image diagnostic mode."}
        else:
            report["conditioning"] = stage("conditioning", lambda: condition_sketch(args.input, directory, cfg["size"]))
            stage("image_generation", lambda: worker("generate", directory))
            report["generation"] = load_json(directory / "generation.json")
        report["preparation"] = stage("foreground_preparation", lambda: prepare_foreground(directory / "03_product.png", directory))
        stage("3d_reconstruction_and_export", lambda: worker("reconstruct", directory))
        report["reconstruction"] = load_json(directory / "reconstruction.json")
        report.update(status="success", stage="complete", total_time_s=round(time.perf_counter() - start, 3), glb="model.glb")
        save_json(report_path, report)
        bundle_run(directory)
        print(f"\\nSUCCESS: {directory / 'model.glb'}", flush=True)
        return 0
    except BaseException as error:
        report.update(status="failed", error=f"{type(error).__name__}: {error}", total_time_s=round(time.perf_counter() - start, 3))
        save_json(report_path, report)
        (directory / "failure_traceback.txt").write_text(traceback.format_exc(), encoding="utf-8")
        bundle_run(directory)
        print(f"\\nFAILED during {report['stage']}: {error}\\nReport: {report_path}", flush=True)
        return 1


def parser():
    p = argparse.ArgumentParser(description=__doc__)
    commands = p.add_subparsers(dest="command", required=True)
    run = commands.add_parser("run", help="Generate a 3D asset from a sketch.")
    run.add_argument("--input", required=True)
    run.add_argument("--run-dir", required=True, help="A new, unused output directory.")
    run.add_argument("--triposr-dir", required=True)
    run.add_argument("--prompt", default="")
    run.add_argument("--seed", type=int, default=42)
    run.add_argument("--quality", choices=PRESETS, default="preview")
    run.add_argument("--candidates", type=int, choices=[1, 2], default=1)
    run.add_argument("--control-strength", type=float, default=1.1)
    run.add_argument("--texture", action="store_true")
    run.add_argument("--require-texture", action="store_true")
    run.add_argument("--obj", action="store_true")
    run.add_argument("--target-faces", type=int, default=0)
    run.add_argument("--product-image", action="store_true", help="Explicit image-to-3D diagnostic; skips diffusion.")
    for name in ("generate", "reconstruct"):
        sub = commands.add_parser(name, help=argparse.SUPPRESS)
        sub.add_argument("--job", required=True)
    return p


def main():
    args = parser().parse_args()
    if args.command == "run":
        if args.require_texture and not args.texture:
            raise SystemExit("--require-texture also needs --texture.")
        if args.target_faces < 0 or not 0.3 <= args.control_strength <= 2.0:
            raise SystemExit("Use nonnegative target faces and sketch strength between 0.3 and 2.0.")
        raise SystemExit(run_pipeline(args))
    if args.command == "generate":
        generate_worker(args.job)
    elif args.command == "reconstruct":
        reconstruct_worker(args.job)


if __name__ == "__main__":
    main()
'''
(ROOT / 'engine.py').write_text(FILE_CONTENT, encoding='utf-8')
print('Wrote engine.py')


Wrote engine.py


In [4]:
# Embedded project file: bootstrap.py
FILE_CONTENT = '''"""Prepare an isolated Linux/NVIDIA environment without changing Colab's kernel."""
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys

ROOT = Path(__file__).resolve().parent
VENV = ROOT / ".venv"
PYTHON = VENV / "bin" / "python"
UV_VERSION = "0.8.22"
PYTHON_VERSION = "3.11.13"
COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"


def run(command, **kwargs):
    print("Running:", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), check=True, **kwargs)


def main():
    if platform.system() != "Linux" or platform.machine() not in ("x86_64", "AMD64"):
        raise RuntimeError("This setup targets Linux x86_64 with an NVIDIA GPU, including Google Colab.")
    try:
        run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
    except (FileNotFoundError, subprocess.CalledProcessError) as error:
        raise RuntimeError("No NVIDIA GPU is attached. Select a GPU runtime in Colab, then rerun this cell.") from error
    if shutil.disk_usage(ROOT).free < 10 * 1024 ** 3:
        raise RuntimeError("Free at least 10 GiB of local disk space for the environment and model downloads.")
    environment = dict(os.environ)
    environment.pop("PYTHONPATH", None)
    environment.update(UV_CACHE_DIR=str(ROOT / ".uv-cache"), UV_PYTHON_INSTALL_DIR=str(ROOT / ".managed-python"), HF_HOME=str(ROOT / "cache" / "huggingface"), U2NET_HOME=str(ROOT / "cache" / "rembg"), TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_TELEMETRY="1")
    tools_dir = ROOT / ".bootstrap"
    if not (tools_dir / "uv").is_dir():
        run([sys.executable, "-m", "pip", "install", "--quiet", "--target", tools_dir, f"uv=={UV_VERSION}"])
    uv_environment = dict(environment, PYTHONPATH=str(tools_dir))
    uv = [sys.executable, "-m", "uv"]
    if not PYTHON.is_file():
        run(uv + ["venv", "--python", PYTHON_VERSION, VENV], env=uv_environment)
    lock = ROOT / "requirements.lock"
    fingerprint = hashlib.sha256(lock.read_bytes()).hexdigest()
    marker = ROOT / "setup_state.json"
    previous = json.loads(marker.read_text()) if marker.is_file() else {}
    if previous.get("requirements_sha256") != fingerprint:
        run(uv + ["pip", "sync", "--python", PYTHON, lock], env=uv_environment)
    run(uv + ["pip", "check", "--python", PYTHON], env=uv_environment)
    upstream = ROOT / "TripoSR"
    if not (upstream / ".git").is_dir():
        upstream.mkdir(exist_ok=True)
        run(["git", "init", "--quiet", upstream])
        run(["git", "-C", upstream, "remote", "add", "origin", "https://github.com/VAST-AI-Research/TripoSR.git"])
    revision = subprocess.check_output(["git", "-C", str(upstream), "rev-parse", "--verify", "HEAD"], text=True, stderr=subprocess.DEVNULL).strip() if (upstream / "tsr").is_dir() else ""
    if revision != COMMIT:
        run(["git", "-C", upstream, "fetch", "--depth", "1", "origin", COMMIT])
        run(["git", "-C", upstream, "checkout", "--quiet", "--detach", "FETCH_HEAD"])
    smoke = """
import sys, torch, cv2, numpy, rembg, xatlas, mcubes
from diffusers import StableDiffusionControlNetPipeline
from transformers import CLIPTextModel
import engine
engine.install_cpu_marching_cubes()
sys.path.insert(0, 'TripoSR')
from tsr.system import TSR
from tsr.models.tokenizers.image import DINOSingleImageTokenizer
from tsr.models.tokenizers.triplane import Triplane1DTokenizer
from tsr.models.transformer.transformer_1d import Transformer1D
from tsr.models.network_utils import TriplaneUpsampleNetwork, NeRFMLP
from tsr.models.nerf_renderer import TriplaneNeRFRenderer
assert torch.cuda.is_available(), 'CUDA is not available in the isolated environment'
assert float(torch.ones(8, device='cuda').sum()) == 8.0
print('Imports and an actual CUDA operation passed on', torch.cuda.get_device_name(0))
"""
    run([PYTHON, "-c", smoke], cwd=ROOT, env=environment)
    with open(ROOT / "environment-resolved.txt", "w") as stream:
        run(uv + ["pip", "freeze", "--python", PYTHON], env=uv_environment, stdout=stream)
    marker.write_text(json.dumps({"requirements_sha256": fingerprint, "python": PYTHON_VERSION, "triposr_commit": COMMIT}, indent=2))
    print("Setup passed. Model weights will download when you first press Generate.")


if __name__ == "__main__":
    main()
'''
(ROOT / 'bootstrap.py').write_text(FILE_CONTENT, encoding='utf-8')
print('Wrote bootstrap.py')


Wrote bootstrap.py


In [5]:
# Embedded project file: notebook_app.py
FILE_CONTENT = '''"""Notebook-native UI. Run after ROOT, PYTHON, and RUN_ENV have been configured."""
import base64
import csv
import datetime
import html
import io
import json
import os
from pathlib import Path
import signal
import subprocess
import uuid
import zipfile

import ipywidgets as widgets
from IPython.display import HTML, JSON, display, clear_output
from PIL import Image, ImageDraw
from google.colab import output, files

output.enable_custom_widget_manager()
state = {"drawing": None, "last_run": None, "busy": False}
input_source = widgets.ToggleButtons(options=["Upload", "Drawing"], description="Input:")
upload = widgets.FileUpload(accept="image/png,image/jpeg,image/webp", multiple=False, description="Upload sketch")
description = widgets.Textarea(placeholder="Optional: e.g. ceramic mug with a large blue handle", description="Describe:", layout=widgets.Layout(width="95%", height="70px"))
quality = widgets.Dropdown(options=[("Preview: start here", "preview"), ("Balanced", "balanced"), ("Detailed: more memory", "detailed")], value="preview", description="Quality:")
seed = widgets.IntText(value=42, description="Seed:")
strength = widgets.FloatSlider(value=1.1, min=0.5, max=1.5, step=0.1, description="Sketch guide:", readout_format=".1f")
candidates = widgets.IntSlider(value=1, min=1, max=2, description="Candidates:")
texture = widgets.Checkbox(value=False, description="Bake UV texture (slower)")
require_texture = widgets.Checkbox(value=False, description="Fail if requested texture cannot be baked")
obj_export = widgets.Checkbox(value=False, description="Also export OBJ + material files")
generate = widgets.Button(description="Generate 3D model", button_style="primary", icon="cube", layout=widgets.Layout(width="220px"))
download = widgets.Button(description="Download last result ZIP", icon="download", disabled=True)
save_drive = widgets.Button(description="Save last result to Drive", icon="save", disabled=True)
message = widgets.HTML("Upload a sketch or use the drawing canvas below. Generation starts only when you press the button.")
results_output = widgets.Output()


def current_upload_path():
    value = upload.value
    items = list(value.values()) if isinstance(value, dict) else list(value)
    if not items:
        raise ValueError("Upload a sketch first, or select Drawing.")
    data = bytes(items[0]["content"])
    if len(data) > 20 * 1024 ** 2:
        raise ValueError("Use an image smaller than 20 MB.")
    destination = ROOT / "inputs" / (uuid.uuid4().hex + ".png")
    destination.parent.mkdir(exist_ok=True)
    with Image.open(io.BytesIO(data)) as image:
        if image.width * image.height > 25_000_000:
            raise ValueError("Resize this image to fewer than 25 million pixels.")
        image.save(destination, format="PNG")
    return destination


def save_drawing(data_url):
    if not isinstance(data_url, str) or not data_url.startswith("data:image/png;base64,") or len(data_url) > 4_000_000:
        raise ValueError("The drawing data is invalid.")
    destination = ROOT / "inputs" / ("drawing_" + uuid.uuid4().hex + ".png")
    destination.parent.mkdir(exist_ok=True)
    data = base64.b64decode(data_url.split(",", 1)[1], validate=True)
    with Image.open(io.BytesIO(data)) as image:
        image.convert("RGB").save(destination)
    state["drawing"] = destination
    input_source.value = "Drawing"
    message.value = "Drawing saved. You can now generate the 3D model."
    return JSON({"saved": True})


output.register_callback("sketchforge.save_drawing", save_drawing)


def viewer_html(glb_path):
    payload = base64.b64encode(Path(glb_path).read_bytes()).decode("ascii")
    return \'\'\'<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<script type="module" src="https://unpkg.com/@google/model-viewer@4.0.0/dist/model-viewer.min.js"></script>
<style>body{margin:0;background:#f5f6f8;font-family:system-ui}model-viewer{width:100%;height:460px}p{padding:0 16px;color:#334155}</style></head>
<body><model-viewer src="data:model/gltf-binary;base64,\'\'\' + payload + \'\'\'" camera-controls auto-rotate shadow-intensity="0.4" exposure="1" alt="The final exported GLB model"></model-viewer>
<p>Drag to rotate; scroll to zoom. This viewer loads the exported GLB. Its viewer script requires internet access.</p></body></html>\'\'\'


def rebuild_result_bundle(directory):
    destination = directory / "result_bundle.zip"
    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(directory.iterdir()):
            if path.is_file() and path != destination:
                archive.write(path, arcname=path.name)
    return destination


def show_result(directory):
    report = json.loads((directory / "report.json").read_text())
    print("Status:", report["status"])
    if report["status"] != "success":
        print("Failed stage:", report.get("stage"), "\\n", report.get("error", ""))
        print("The diagnostic ZIP includes the report, intermediate images, and logs.")
        return
    reconstruction = report["reconstruction"]
    print("Backend: TripoSR | Appearance:", reconstruction["appearance"], "| Time:", report["total_time_s"], "seconds")
    for warning in reconstruction["warnings"]:
        print("Note:", warning)
    for name, label in [("00_input.png", "Input"), ("02_control.png", "Sketch guidance"), ("03_product.png", "Generated object"), ("04_foreground.png", "Isolated object")]:
        if not (directory / name).is_file():
            continue
        print(label)
        with Image.open(directory / name) as image:
            display(image.copy().resize((280, 280)))
    viewer = viewer_html(directory / "model.glb")
    (directory / "viewer.html").write_text(viewer)
    display(HTML('<iframe title="Exported 3D model" style="width:100%;height:530px;border:1px solid #ddd;border-radius:8px" srcdoc="' + html.escape(viewer, quote=True) + '"></iframe>'))
    print("Export checks:")
    display(JSON(reconstruction["export_validation"]))
    rebuild_result_bundle(directory)


def run_case(image_path, prompt="", quality_name="preview", seed_value=42, bake=False, strict_texture=False, number_candidates=1, sketch_strength=1.1, export_obj=False, show=True, product_image=False):
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
    directory = ROOT / "runs" / (stamp + "_" + uuid.uuid4().hex[:8])
    directory.parent.mkdir(exist_ok=True)
    command = [str(PYTHON), str(ROOT / "engine.py"), "run", "--input", str(image_path), "--run-dir", str(directory), "--triposr-dir", str(ROOT / "TripoSR"), "--prompt", str(prompt), "--seed", str(seed_value), "--quality", quality_name, "--candidates", str(number_candidates), "--control-strength", str(sketch_strength)]
    for enabled, flag in [(bake, "--texture"), (strict_texture, "--require-texture"), (export_obj, "--obj"), (product_image, "--product-image")]:
        if enabled:
            command.append(flag)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=RUN_ENV, start_new_session=True)
    captured = []
    try:
        for line in process.stdout:
            print(line, end="")
            captured.append(line)
        process.wait()
    except BaseException:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise
    if directory.is_dir():
        (directory / "notebook_execution.log").write_text("".join(captured))
    if not (directory / "report.json").is_file():
        raise RuntimeError("The pipeline could not create a report. Check the output above for an environment error.")
    state["last_run"] = directory
    download.disabled = False
    save_drive.disabled = False
    rebuild_result_bundle(directory)
    if show:
        show_result(directory)
    return directory


def on_generate(_):
    if state["busy"]:
        return
    state["busy"] = True
    generate.disabled = True
    with results_output:
        clear_output(wait=True)
        try:
            if require_texture.value and not texture.value:
                raise ValueError("Enable UV texture baking before requiring a texture.")
            path = current_upload_path() if input_source.value == "Upload" else state["drawing"]
            if path is None:
                raise ValueError("Press Use drawing on the canvas first.")
            directory = run_case(path, prompt=description.value, quality_name=quality.value, seed_value=seed.value, bake=texture.value, strict_texture=require_texture.value, number_candidates=candidates.value, sketch_strength=strength.value, export_obj=obj_export.value)
            status = json.loads((directory / "report.json").read_text())["status"]
            message.value = "Finished. Download the result ZIP below." if status == "success" else "This run failed. The saved report explains which stage needs attention."
        except Exception as error:
            print(type(error).__name__ + ":", error)
        finally:
            state["busy"] = False
            generate.disabled = False


def on_download(_):
    if state["last_run"]:
        files.download(str(rebuild_result_bundle(state["last_run"])))


def on_save_drive(_):
    import shutil
    from google.colab import drive
    with results_output:
        drive.mount("/content/drive")
        destination = Path("/content/drive/MyDrive/SketchForge3D_results")
        destination.mkdir(exist_ok=True)
        archive = rebuild_result_bundle(state["last_run"])
        output_path = destination / (state["last_run"].name + ".zip")
        shutil.copy2(archive, output_path)
        print("Saved:", output_path)


generate.on_click(on_generate)
download.on_click(on_download)
save_drive.on_click(on_save_drive)
advanced = widgets.Accordion(children=[widgets.VBox([seed, strength, candidates, texture, require_texture, obj_export])])
advanced.set_title(0, "Optional settings")
advanced.selected_index = None
display(widgets.VBox([input_source, upload, description, quality, advanced, generate, message, widgets.HBox([download, save_drive]), results_output]))

DRAWING_HTML = r\'\'\'<div id="sketchforge-drawing" style="font:14px system-ui;max-width:540px;margin:16px 0">
<p><strong>Drawing canvas</strong> — draw one complete object.</p>
<canvas width="512" height="512" style="border:1px solid #9ca3af;border-radius:6px;width:100%;touch-action:none"></canvas>
<div style="display:flex;gap:8px;align-items:center;margin-top:8px"><button data-action="pen">Pen</button><button data-action="erase">Eraser</button><button data-action="clear">Clear</button><label>Size <input type="range" min="2" max="24" value="5"></label><button data-action="save">Use drawing</button></div><p data-status></p>
<script>(function(){const root=document.getElementById('sketchforge-drawing');const c=root.querySelector('canvas');const ctx=c.getContext('2d');let drawing=false;let erase=false;ctx.fillStyle='white';ctx.fillRect(0,0,512,512);ctx.lineCap='round';ctx.lineJoin='round';
function pos(e){const r=c.getBoundingClientRect();return [(e.clientX-r.left)*512/r.width,(e.clientY-r.top)*512/r.height]}
c.onpointerdown=e=>{drawing=true;c.setPointerCapture(e.pointerId);const p=pos(e);ctx.strokeStyle=erase?'white':'black';ctx.lineWidth=Number(root.querySelector('input').value)*(erase?3:1);ctx.beginPath();ctx.moveTo(p[0],p[1]);ctx.lineTo(p[0]+0.1,p[1]+0.1);ctx.stroke()};
c.onpointermove=e=>{if(!drawing)return;const p=pos(e);ctx.lineTo(p[0],p[1]);ctx.stroke()};c.onpointerup=()=>{drawing=false};c.onpointercancel=()=>{drawing=false};
root.querySelector('[data-action="pen"]').onclick=()=>{erase=false};root.querySelector('[data-action="erase"]').onclick=()=>{erase=true};root.querySelector('[data-action="clear"]').onclick=()=>{ctx.fillStyle='white';ctx.fillRect(0,0,512,512)};
root.querySelector('[data-action="save"]').onclick=async()=>{const status=root.querySelector('[data-status]');try{await google.colab.kernel.invokeFunction('sketchforge.save_drawing',[c.toDataURL('image/png')],{});status.textContent='Drawing saved and selected. Press Generate 3D model above.'}catch(e){status.textContent='Could not save: '+e.message}};
})();</script></div>\'\'\'
display(HTML(DRAWING_HTML))


def evaluate_cases(cases, quality_name="preview"):
    """Explicit, small evaluation: cases = [(image_path, optional_prompt), ...]."""
    if not cases or len(cases) > 12:
        raise ValueError("Evaluate between 1 and 12 examples at a time on free Colab.")
    rows = []
    for number, (path, prompt) in enumerate(cases, 1):
        print(f"\\nEvaluation example {number}/{len(cases)}: {Path(path).name}")
        directory = run_case(path, prompt=prompt, quality_name=quality_name, show=False)
        report = json.loads((directory / "report.json").read_text())
        reconstruction = report.get("reconstruction", {})
        rows.append({"input": Path(path).name, "prompt_provided": bool(prompt.strip()), "status": report["status"], "backend": reconstruction.get("backend", ""), "glb_passed": reconstruction.get("export_validation", {}).get("passed", False), "appearance": reconstruction.get("appearance", ""), "total_time_s": report["total_time_s"], "run_dir": str(directory), "error": report.get("error", "")})
    destination = ROOT / "evaluation.csv"
    with destination.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print("Saved:", destination)
    return rows
'''
(ROOT / 'notebook_app.py').write_text(FILE_CONTENT, encoding='utf-8')
print('Wrote notebook_app.py')


Wrote notebook_app.py


In [6]:
# Embedded project file: test_engine.py
FILE_CONTENT = '''"""CPU checks for the actual pipeline helpers; no generated examples are fabricated."""
import json
from pathlib import Path
import tempfile
import types
import unittest
import zipfile

import numpy as np
from PIL import Image, ImageDraw
import trimesh
import engine


class PipelineChecks(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.root = Path(self.temp.name)

    def tearDown(self):
        self.temp.cleanup()

    def sketch(self, inverted=False):
        image = Image.new("RGB", (400, 240), "black" if inverted else "white")
        draw = ImageDraw.Draw(image)
        draw.rectangle((80, 40, 300, 180), outline="white" if inverted else "black", width=4)
        path = self.root / ("inverted.png" if inverted else "sketch.png")
        image.save(path)
        return path

    def test_blank_is_rejected(self):
        source = self.root / "blank.png"
        Image.new("RGB", (64, 64), "white").save(source)
        with self.assertRaisesRegex(ValueError, "blank"):
            engine.condition_sketch(source, self.root)

    def test_control_map_shape_and_structure(self):
        stats = engine.condition_sketch(self.sketch(), self.root)
        control = np.asarray(Image.open(self.root / "02_control.png"))
        self.assertEqual(control.shape, (512, 512, 3))
        self.assertGreater(stats["stroke_fraction"], 0)
        self.assertLess(stats["stroke_fraction"], 0.2)
        # Framing preserves the rectangular proportions rather than stretching.
        y, x = np.nonzero(control[..., 0])
        self.assertAlmostEqual((x.max() - x.min()) / (y.max() - y.min()), 220 / 140, delta=0.07)

    def test_inverse_sketch_has_same_structure(self):
        a, b = self.root / "a", self.root / "b"
        a.mkdir()
        b.mkdir()
        engine.condition_sketch(self.sketch(), a)
        result = engine.condition_sketch(self.sketch(True), b)
        self.assertTrue(result["inverted_input"])
        np.testing.assert_array_equal(np.asarray(Image.open(a / "02_control.png")), np.asarray(Image.open(b / "02_control.png")))

    def test_transparent_background_is_white(self):
        image = Image.new("RGBA", (20, 20), (0, 0, 0, 0))
        image.putpixel((10, 10), (0, 0, 0, 255))
        source = self.root / "alpha.png"
        image.save(source)
        rgb = np.asarray(engine.read_rgb(source))
        np.testing.assert_array_equal(rgb[0, 0], [255, 255, 255])
        np.testing.assert_array_equal(rgb[10, 10], [0, 0, 0])

    def test_small_disconnected_components_survive(self):
        body = trimesh.creation.box()
        detail = trimesh.creation.icosphere(subdivisions=1, radius=0.03)
        detail.apply_translation([0.7, 0, 0])
        joined = trimesh.util.concatenate([body, detail])
        cleaned = engine.clean_mesh(joined)
        self.assertEqual(len(cleaned.split(only_watertight=False)), 2)
        self.assertEqual(len(cleaned.faces), len(joined.faces))

    def test_simple_real_3d_geometry_is_accepted(self):
        report = engine.geometry_report(trimesh.creation.box())
        self.assertTrue(report["passed"])
        self.assertEqual(report["faces"], 12)

    def test_optional_decimation_keeps_geometry_and_colors(self):
        mesh = trimesh.creation.icosphere(subdivisions=3)
        mesh.visual.vertex_colors = np.tile([70, 140, 210, 255], (len(mesh.vertices), 1))
        reduced = engine.clean_mesh(mesh, target_faces=300)
        self.assertLess(len(reduced.faces), len(mesh.faces))
        self.assertTrue(engine.geometry_report(reduced)["passed"])
        self.assertEqual(reduced.visual.kind, "vertex")
        self.assertEqual(len(reduced.visual.vertex_colors), len(reduced.vertices))

    def test_flat_output_is_rejected(self):
        plane = trimesh.Trimesh(vertices=[[0, 0, 0], [1, 0, 0], [0, 1, 0]], faces=[[0, 1, 2]], process=False)
        self.assertFalse(engine.geometry_report(plane)["passed"])

    def test_vertex_colors_survive_glb_round_trip(self):
        mesh = trimesh.creation.box()
        mesh.visual.vertex_colors = np.tile([20, 100, 210, 255], (len(mesh.vertices), 1))
        path = self.root / "colored.glb"
        mesh.export(path)
        check = engine.inspect_glb(path, expect_vertex_colors=True)
        self.assertTrue(check["passed"])
        self.assertTrue(check["has_vertex_colors"])
        self.assertFalse(engine.inspect_glb(path, expect_texture=True)["passed"])

    def test_uv_rasterization_matches_texture_coordinates(self):
        vertices = np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0]], dtype=np.float32)
        positions, mask = engine.rasterize_uv_positions(vertices, np.array([[0, 1, 2]]), vertices[:, :2], 32)
        ys, xs = np.nonzero(mask)
        np.testing.assert_allclose(positions[mask, 0], (xs + 0.5) / 32, atol=1e-6)
        np.testing.assert_allclose(positions[mask, 1], 1 - (ys + 0.5) / 32, atol=1e-6)

    def test_texture_bake_and_export_without_opengl(self):
        mesh = trimesh.creation.box()
        textured, texture = engine.bake_texture_cpu_atlas(mesh, 64, lambda points: points + 0.5)
        self.assertEqual(texture.size, (64, 64))
        self.assertEqual(textured.visual.kind, "texture")
        np.testing.assert_allclose(textured.bounds, mesh.bounds)
        path = self.root / "textured.glb"
        textured.export(path)
        check = engine.inspect_glb(path, expect_texture=True)
        self.assertTrue(check["passed"])
        self.assertTrue(check["has_texture"])
        engine.write_obj(textured, self.root)
        self.assertTrue((self.root / "model.obj").is_file())
        self.assertTrue((self.root / "model.mtl").is_file())
        self.assertGreater(len(list(self.root.glob("*.png"))), 0)

    def test_marching_cubes_adapter_preserves_asymmetric_axes(self):
        import mcubes
        xx, yy, zz = np.meshgrid(np.arange(24), np.arange(28), np.arange(32), indexing="ij")
        volume = ((xx - 7) / 3) ** 2 + ((yy - 12) / 5) ** 2 + ((zz - 18) / 7) ** 2
        expected, _ = mcubes.marching_cubes(volume, 1.0)
        adapted, faces = engine.marching_cubes_numpy(volume, 1.0)
        # The upstream helper reverses the adapter output's axis order.
        recovered = adapted[:, [2, 1, 0]]
        np.testing.assert_allclose(recovered.min(axis=0), expected.min(axis=0), atol=1e-5)
        np.testing.assert_allclose(recovered.max(axis=0), expected.max(axis=0), atol=1e-5)
        self.assertGreater(len(faces), 0)

    def test_failed_run_has_failed_report_and_diagnostic_bundle(self):
        source = self.root / "blank.png"
        Image.new("RGB", (64, 64), "white").save(source)
        output = self.root / "failed_run"
        args = types.SimpleNamespace(run_dir=str(output), quality="preview", prompt="", seed=42, candidates=1, control_strength=1.1, texture=False, require_texture=False, target_faces=0, obj=False, triposr_dir=str(self.root), input=str(source), product_image=False)
        self.assertEqual(engine.run_pipeline(args), 1)
        report = json.loads((output / "report.json").read_text())
        self.assertEqual(report["status"], "failed")
        self.assertNotIn("glb", report)
        with zipfile.ZipFile(output / "result_bundle.zip") as archive:
            self.assertIn("report.json", archive.namelist())
            self.assertNotIn("model.glb", archive.namelist())


if __name__ == "__main__":
    unittest.main(verbosity=2)
'''
(ROOT / 'test_engine.py').write_text(FILE_CONTENT, encoding='utf-8')
print('Wrote test_engine.py')


Wrote test_engine.py


In [7]:
# Embedded project file: requirements.in
FILE_CONTENT = '''torch==2.6.0
torchvision==0.21.0
numpy==1.26.4
Pillow==10.4.0
opencv-python-headless==4.10.0.84
scipy==1.14.1
diffusers==0.30.3
transformers==4.44.2
accelerate==0.34.2
huggingface-hub==0.25.2
safetensors==0.4.5
tokenizers==0.19.1
omegaconf==2.3.0
einops==0.8.0
rembg[cpu]==2.0.60
onnxruntime==1.19.2
numba==0.60.0
llvmlite==0.43.0
scikit-image==0.24.0
trimesh==4.5.3
PyMCubes==0.1.6
xatlas==0.0.11
fast-simplification==0.1.7
psutil==6.1.0
tqdm==4.66.6
'''
(ROOT / 'requirements.in').write_text(FILE_CONTENT, encoding='utf-8')
print('Wrote requirements.in')


Wrote requirements.in


In [8]:
# Embedded project file: requirements.lock
FILE_CONTENT = '''accelerate==0.34.2
antlr4-python3-runtime==4.9.3
attrs==26.1.0
certifi==2026.10.23
charset-normalizer==3.5.2
coloredlogs==15.0.1
diffusers==0.30.3
einops==0.8.0
fast-simplification==0.1.7
filelock==4.2.6
flatbuffers==25.12.19
fsspec==2026.9.0
huggingface-hub==0.25.2
humanfriendly==10.0
idna==3.20
imageio==2.38.1
importlib-metadata==10.1.0
jinja2==3.1.6
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
lazy-loader==0.6
llvmlite==0.43.0
markupsafe==3.0.4
mpmath==1.3.0
networkx==3.6.1
numba==0.60.0
numpy==1.26.4
nvidia-cublas-cu12==12.4.5.8
nvidia-cuda-cupti-cu12==12.4.127
nvidia-cuda-nvrtc-cu12==12.4.127
nvidia-cuda-runtime-cu12==12.4.127
nvidia-cudnn-cu12==9.1.0.70
nvidia-cufft-cu12==11.2.1.3
nvidia-curand-cu12==10.3.5.147
nvidia-cusolver-cu12==11.6.1.9
nvidia-cusparse-cu12==12.3.1.170
nvidia-cusparselt-cu12==0.6.2
nvidia-nccl-cu12==2.21.5
nvidia-nvjitlink-cu12==12.4.127
nvidia-nvtx-cu12==12.4.127
omegaconf==2.3.0
onnxruntime==1.19.2
opencv-python-headless==4.10.0.84
packaging==26.3
pillow==10.4.0
platformdirs==4.14.2
pooch==1.9.0
protobuf==7.36.2
psutil==6.1.0
pymatting==1.1.16
pymcubes==0.1.6
pyyaml==6.0.3
referencing==0.37.0
regex==2026.9.29
rembg==2.0.60
requests==2.34.2
rpds-py==2026.9.1
safetensors==0.4.5
scikit-image==0.24.0
scipy==1.14.1
sympy==1.13.1
tifffile==2026.3.3
tokenizers==0.19.1
torch==2.6.0
torchvision==0.21.0
tqdm==4.66.6
transformers==4.44.2
trimesh==4.5.3
triton==3.2.0
typing-extensions==4.16.0
urllib3==2.8.0
xatlas==0.0.11
zipp==4.1.2
'''
(ROOT / 'requirements.lock').write_text(FILE_CONTENT, encoding='utf-8')
print('Wrote requirements.lock')


Wrote requirements.lock


In [9]:
# Embedded project file: README.md
FILE_CONTENT = '''# SketchForge 3D — complete Colab implementation

Convert a single object sketch into a generated object image, then a genuine TripoSR mesh, with GLB export and optional UV texture baking.

## Start in Google Colab

1. Open https://colab.research.google.com/ and choose **File → Upload notebook**.
2. Upload `SketchForge3D_Colab_Complete.ipynb`.
3. Select **Runtime → Change runtime type → GPU**. Free GPU access is subject to availability.
4. Run the notebook cells in order, or choose **Run all**. Setup creates an isolated Python environment; it does not replace the Colab kernel's packages. Several GB of downloads and several minutes of setup may be needed.
5. In the interface, upload a sketch or draw on the canvas and press **Use drawing**.
6. Optionally describe the object. Start with **Preview** quality and press **Generate 3D model**.
7. Inspect the exported model in the viewer, then download the result ZIP. Try **Balanced** quality and **Bake UV texture** after your first successful run.

Running all cells prepares the interface and runs small CPU checks. It does **not** start a GPU generation job, mount Google Drive, or launch a dataset batch. Those actions need their corresponding controls or explicitly enabled cells.

## Included implementation

- Stable Diffusion 1.5 + ControlNet Scribble through Diffusers; the model's default safety checker stays enabled.
- Upload and drawing inputs with an explicit input selector, optional text, seed and quality controls.
- Sketch framing without stretching; support for black-on-white and white-on-black drawings.
- CPU background removal with IS-Net through rembg. Small object parts are retained.
- TripoSR reconstruction in float32 with a CPU PyMCubes extraction adapter.
- Process separation between neural stages: worker exit releases GPU allocations.
- Optional CPU UV rasterization through xatlas and chunked GPU color queries. OpenGL/EGL is not needed.
- Conservative cleanup, GLB export/reload validation, vertex colors or verified embedded texture.
- Optional OBJ export with material files.
- An interactive viewer that loads the final exported GLB, rather than the neural field.
- Per-stage logs, immutable model revisions, a full Linux dependency lock, seeds, timings, GPU statistics, checksums, failure reports and downloadable bundles.
- Optional small evaluation and Google Drive saving, both initiated explicitly.
- A command-line interface and CPU regression checks.

## Files and architecture

The notebook embeds the complete project source. It materializes it under `/content/sketchforge3d` when run. The companion source ZIP is provided for inspection and command-line use on Linux/NVIDIA hardware.

`engine.py` is the CLI and inference pipeline. `bootstrap.py` creates the isolated environment and checks imports and a real CUDA operation. `notebook_app.py` contains the notebook-native interface. `test_engine.py` contains CPU checks. `requirements.lock` pins the Linux/Python 3.11 dependency set; `requirements.in` records the selected primary versions.

The image-generation worker exits before the reconstruction worker starts. Intermediate images on disk connect the stages. Repeated runs reload models from the local download cache; this trades some loading time for predictable GPU memory use.

## Command-line use

On Linux x86_64 with an NVIDIA GPU, unpack the source ZIP, run `python bootstrap.py`, then use the generated environment:

```bash
.venv/bin/python engine.py run --input sketch.png --run-dir runs/example1 --triposr-dir TripoSR --prompt "blue ceramic mug" --quality preview
```

Use a new output directory for each run. Add `--texture` for UV baking, `--require-texture` to fail if the requested bake fails, and `--obj` for an additional OBJ bundle. Run `engine.py run --help` for other options.

For an explicit image-to-3D diagnostic, supply a product image and `--product-image`. This is recorded as skipped diffusion, not counted as a full sketch-to-3D run.

## Result bundle

Each run contains original/framed sketches, the control map, generated image candidates, selected object image, foreground/mask, reconstruction input, `model.glb`, optional texture and OBJ files, configuration, stage reports, logs and `report.json`. The notebook adds `viewer.html`. The ZIP records failures as failures and includes diagnostic files.

The GLB is the authoritative validated output. A successful reload confirms actual triangle geometry and the expected appearance representation. Watertightness is reported, but manufacturing readiness is not claimed. Inspect the silhouette, important parts and unseen surfaces yourself.

## Reproducibility and resources

The setup uses managed Python 3.11.13, PyTorch 2.6.0 and a locked package set in a separate environment. It checks out TripoSR commit `107cefdc244c39106fa830359024f6a2f1c78871`. The source model, ControlNet, TripoSR checkpoint and DINO configuration have fixed Hugging Face revisions recorded in `engine.py` and every run report.

Pretrained model weights are downloaded at runtime, not bundled. The rembg library version fixes its checkpoint configuration and checksum expectations. Record `environment-resolved.txt` and reports with a submission. The same seed does not imply identical results across different GPU/software environments.

A GPU runtime and sufficient free disk space are required. Preview settings are designed as a starting point for a free Colab GPU, not a guarantee of runtime or availability. See https://research.google.com/colaboratory/faq.html for resource and notebook-interface restrictions.

## Validation status

The notebook format, Python cell syntax, preprocessing, UV rasterization/baking, mesh checks, GLB/texture round trips, and failure reports are tested locally. The full Linux dependency set has been resolved. Full neural inference and Colab's interactive UI still require verification in a GPU Colab session; no GPU-generated results or benchmark timings are fabricated.

## Troubleshooting

- **No GPU:** attach a GPU runtime; free availability varies. Setup stops before downloading model weights if none is attached.
- **Setup download error:** rerun setup; the environment is separate from Colab's built-in packages. Do not randomly upgrade the locked packages.
- **Out of GPU memory:** use Preview, one candidate and no texture. Interrupt any other GPU work. Restart if an earlier external notebook retained GPU memory.
- **Background removal error:** inspect the selected object image. Try a complete object with a plain background and a more specific optional description.
- **Wrong object or missing parts:** inspect the control map, then the generated image and mask before diagnosing the 3D model. Text is optional, but ambiguous drawings remain difficult.
- **Texture baking failure:** a non-strict run keeps vertex colors and records the failure. Enable the strict texture option when evaluating actual texture generation.
- **Viewer unavailable:** download `model.glb` and open it in Blender. The embedded viewer needs internet access to load its pinned model-viewer script.
- **Interrupted session:** saved files under `/content` can disappear when the runtime is replaced. Download result ZIPs or explicitly save them to Drive.

## Models, libraries and licenses

- [Stable Diffusion 1.5 mirror](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5): CreativeML OpenRAIL-M, with use restrictions.
- [ControlNet Scribble](https://huggingface.co/lllyasviel/control_v11p_sd15_scribble): see its OpenRAIL license and model card.
- [TripoSR](https://github.com/VAST-AI-Research/TripoSR), [weights](https://huggingface.co/stabilityai/TripoSR): MIT; retain upstream attribution when distributing upstream code.
- [rembg](https://github.com/danielgatis/rembg): MIT library; inspect the linked IS-Net checkpoint/project terms separately.
- [DINO](https://huggingface.co/facebook/dino-vitb16): configuration used by TripoSR's image tokenizer.
- [PyMCubes](https://github.com/pmneila/PyMCubes), [xatlas bindings](https://github.com/mworchel/xatlas-python), [trimesh](https://github.com/mikedh/trimesh), [Diffusers](https://github.com/huggingface/diffusers), [PyTorch](https://pytorch.org/) and [model-viewer](https://modelviewer.dev/): retain their respective notices when redistributing components.

New integration source is supplied in full. This does not replace or override any pretrained model or third-party library license.
'''
(ROOT / 'README.md').write_text(FILE_CONTENT, encoding='utf-8')
print('Wrote README.md')


Wrote README.md


In [10]:
# Embedded project file: verification.json
FILE_CONTENT = '''{
  "artifact_version": "1.0.0",
  "checked_on": "2026-10-31",
  "cpu_regression_tests": {"passed": 13, "failed": 0},
  "checks": [
    "Notebook v4 schema and Python code-cell syntax",
    "Embedded project files match companion source files",
    "Linux x86_64 / Python 3.11 dependency set resolved: 73 packages",
    "Pinned Python 3.11.13 Linux x86_64 managed distribution available",
    "Selected PyTorch 2.6.0, torchvision 0.21.0, Diffusers and all TripoSR component imports on the local CPU host",
    "Blank and inverted sketch handling, aspect-preserving framing, transparent input",
    "Disconnected component preservation and optional decimation",
    "CPU UV atlas baking, UV orientation, GLB texture/color round trips, OBJ side files",
    "Asymmetric marching-cubes axis convention",
    "Failed runs stay failed and produce diagnostic bundles"
  ],
  "not_verified_here": [
    "Full GPU neural inference",
    "Installation and execution on a live Google Colab GPU runtime",
    "Colab widget and drawing interactions",
    "Generation latency, GPU memory peaks and sketch-to-3D visual fidelity"
  ],
  "notes": "Local tests use Python 3.12 on macOS. The notebook creates a separate locked Python 3.11.13 Linux environment. No model weights or generated results were fabricated."
}
'''
(ROOT / 'verification.json').write_text(FILE_CONTENT, encoding='utf-8')
print('Wrote verification.json')


Wrote verification.json


## 3. Install and verify the isolated environment

This creates a separate Python 3.11 environment with a fully resolved Linux dependency lock. It keeps Colab's built-in packages intact. TripoSR source is fetched at a fixed commit; a CPU marching-cubes adapter avoids compiling a custom CUDA extension.

This cell stops if installation, dependency checks, imports, or an actual CUDA operation fail. If no GPU is available, attach a GPU runtime before continuing. The small CPU tests do not load neural weights.

In [11]:
subprocess.run([sys.executable, str(ROOT / 'bootstrap.py')], check=True)
PYTHON = ROOT / '.venv' / 'bin' / 'python'
RUN_ENV = dict(os.environ)
RUN_ENV.pop('PYTHONPATH', None)
RUN_ENV.update(
    HF_HOME=str(ROOT / 'cache' / 'huggingface'),
    U2NET_HOME=str(ROOT / 'cache' / 'rembg'),
    TOKENIZERS_PARALLELISM='false',
    HF_HUB_DISABLE_TELEMETRY='1',
    PYTHONUNBUFFERED='1',
    OMP_NUM_THREADS='2',
)
subprocess.run([str(PYTHON), '-m', 'unittest', '-v', 'test_engine'], cwd=ROOT, env=RUN_ENV, check=True)
print('Environment and CPU checks passed. No neural generation has started.')


Environment and CPU checks passed. No neural generation has started.


## 4. Open the notebook controls

Choose **Upload** or **Drawing** explicitly. For a drawing, press **Use drawing** on the canvas before generating. A text description is optional; ambiguous sketches may be interpreted incorrectly without one.

Preview uses 512px image generation and a lower mesh resolution. Balanced and Detailed increase reconstruction work. Start with one candidate and vertex colors. Add UV texture baking after a successful run; its strict option requires a real texture to survive GLB export.

Each stage writes its intermediate images and logs. The neural workers run sequentially in separate processes. A TripoSR failure remains a failure; there is no depth-relief substitution.

In [14]:
if not (ROOT / 'setup_state.json').is_file():
    raise RuntimeError('Run the setup cell successfully first.')
exec(compile((ROOT / 'notebook_app.py').read_text(), str(ROOT / 'notebook_app.py'), 'exec'), globals())


## 5. Optional example and reconstruction diagnostic

A simple procedural mug sketch is supplied below for a first test. The cell only writes the sketch unless you explicitly enable the test flag. It is a 2D input example, not a pre-generated 3D result.

For real evaluation, also use fresh human drawings, thin structures and at least one run with an empty description. A successful file check alone does not prove that the object matches the sketch.

In [15]:
from PIL import Image, ImageDraw
sample_dir = ROOT / 'examples'
sample_dir.mkdir(exist_ok=True)
sample = Image.new('RGB', (512, 512), 'white')
pen = ImageDraw.Draw(sample)
pen.ellipse((105, 100, 345, 160), outline='black', width=5)
pen.line([(105, 130), (120, 390)], fill='black', width=5)
pen.line([(345, 130), (330, 390)], fill='black', width=5)
pen.arc((120, 360, 330, 420), 0, 180, fill='black', width=5)
pen.arc((300, 165, 440, 345), 270, 90, fill='black', width=5)
pen.arc((330, 200, 410, 310), 270, 90, fill='black', width=5)
sample_path = sample_dir / 'mug_sketch.png'
sample.save(sample_path)
print('Example sketch:', sample_path)

RUN_EXAMPLE = False  # Set True and run this cell to spend GPU time on the example.
if RUN_EXAMPLE:
    example_run = run_case(sample_path, prompt='a blue ceramic mug with a large handle', quality_name='preview')
    print('Example report:', example_run / 'report.json')

# To isolate reconstruction using an existing product image, call:
# diagnostic_run = run_case('/content/product.png', product_image=True)
# This is recorded as a diagnostic, not a full sketch-to-3D success.


Example sketch: /content/sketchforge3d/examples/mug_sketch.png


## 6. Optional small evaluation

The evaluator processes up to 12 explicitly supplied cases and writes `evaluation.csv`. Include an empty description case. It records failures separately and reports the validated exported GLB and actual backend. Inspect front, side and back views yourself; 2D edge agreement is only a candidate-ranking heuristic.

In [16]:
RUN_EVALUATION = True  # Keep False unless you intentionally want to run these cases.
CASES = [
    (str(sample_path), 'a blue ceramic mug with a large handle'),
    (str(sample_path), ''),  # Required check: generation without a user text prompt.
]
# Add paths to your own fresh sketches to CASES.
if RUN_EVALUATION:
    evaluation_rows = evaluate_cases(CASES, quality_name='preview')
    display(evaluation_rows)
else:
    print('Evaluation is disabled. No batch generation was started.')


Evaluation example 1/2: mug_sketch.png

--- conditioning ---

--- image_generation ---
Loading image generation models; the first run downloads weights.

Fetching 15 files: 100%|██████████| 15/15 [00:12<00:00,  1.20it/s]

Loading pipeline components...:  14%|█▍        | 1/7 [00:00<00:02,  2.17it/s]/content/sketchforge3d/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(

Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  5.77it/s]
Generating candidate 1/1 (seed 42).

100%|██████████| 20/20 [00:05<00:00,  3.37it/s]

--- foreground_preparation ---

  0%|                                               | 0.00/179M [00:00<?, ?B/s]
  3%|█▎        

[{'input': 'mug_sketch.png',
  'prompt_provided': True,
  'status': 'success',
  'backend': 'TripoSR',
  'glb_passed': True,
  'appearance': 'vertex_colors',
  'total_time_s': 70.757,
  'run_dir': '/content/sketchforge3d/runs/20261101_003249_49f2f14a',
  'error': ''},
 {'input': 'mug_sketch.png',
  'prompt_provided': False,
  'status': 'success',
  'backend': 'TripoSR',
  'glb_passed': True,
  'appearance': 'vertex_colors',
  'total_time_s': 56.009,
  'run_dir': '/content/sketchforge3d/runs/20261101_003401_e6434789',
  'error': ''}]

## 7. Download or save results

The interface's **Download last result ZIP** button includes the model, optional texture/materials, intermediate images, configuration, logs, a report, and an HTML viewer after a successful run. Failed runs provide diagnostics and remain marked as failed.

**Save last result to Drive** mounts Drive only when you click it, then copies that result ZIP. The viewer loads the final GLB and needs internet access for the pinned model-viewer script. You can always inspect the downloaded GLB in Blender.

The following cell can create an additional source archive. It does not contain model weights or generated outputs.

In [17]:

from pathlib import Path
import hashlib
import importlib.metadata

PROJECT_ROOT = Path("/content/sketchforge3d")

print("=" * 50)
print("SKETCHFORGE 3D — SAFE PROJECT CHECK")
print("=" * 50)

# File fingerprints from your previously uploaded source ZIP
original_hashes = {
    "engine.py": "6d3285e047e47437",
    "bootstrap.py": "d8f158cc7aae2b5f",
    "notebook_app.py": "0f0816881eea352b",
    "test_engine.py": "dd362a8afd7c455c",
}

print("\n1. ORIGINAL PROJECT FILES")

for filename, expected_hash in original_hashes.items():
    path = PROJECT_ROOT / filename

    if not path.is_file():
        print(f"❌ {filename}: MISSING")
        continue

    actual_hash = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()[:16]

    if actual_hash == expected_hash:
        print(f"✅ {filename}: MATCHES UPLOADED SOURCE")
    else:
        print(f"⚠️ {filename}: DIFFERENT FROM UPLOADED SOURCE")

print("\n2. BACKEND ENVIRONMENT")

venv_python = PROJECT_ROOT / ".venv" / "bin" / "python"

print("Virtual environment:", "FOUND" if venv_python.is_file() else "MISSING")
print("Inference engine:", "FOUND" if (PROJECT_ROOT / "engine.py").is_file() else "MISSING")
print("TripoSR directory:", "FOUND" if (PROJECT_ROOT / "TripoSR").is_dir() else "MISSING")

print("\n3. GRADIO STATUS")

try:
    version = importlib.metadata.version("gradio")
    print("Gradio in Colab kernel:", version)
except importlib.metadata.PackageNotFoundError:
    print("Gradio in Colab kernel: NOT INSTALLED")

gradio_file = PROJECT_ROOT / "gradio_app.py"

print(
    "Standalone Gradio file:",
    "EXISTS" if gradio_file.is_file() else "NOT CREATED"
)

print("\n4. EXISTING GENERATED RESULTS")

runs_dir = PROJECT_ROOT / "runs"

if runs_dir.is_dir():
    runs = [
        p for p in runs_dir.iterdir()
        if p.is_dir()
    ]

    successful = [
        p for p in runs
        if (p / "model.glb").is_file()
    ]

    print("Total run directories:", len(runs))
    print("Runs containing GLB:", len(successful))
else:
    print("Runs directory not found")

print("\n" + "=" * 50)
print("READ-ONLY CHECK COMPLETED")
print("No project files were modified by this cell.")
print("=" * 50)


SKETCHFORGE 3D — SAFE PROJECT CHECK

1. ORIGINAL PROJECT FILES
✅ engine.py: MATCHES UPLOADED SOURCE
✅ bootstrap.py: MATCHES UPLOADED SOURCE
✅ notebook_app.py: MATCHES UPLOADED SOURCE
✅ test_engine.py: MATCHES UPLOADED SOURCE

2. BACKEND ENVIRONMENT
Virtual environment: FOUND
Inference engine: FOUND
TripoSR directory: FOUND

3. GRADIO STATUS
Gradio in Colab kernel: 6.30.0
Standalone Gradio file: NOT CREATED

4. EXISTING GENERATED RESULTS
Total run directories: 2
Runs containing GLB: 2

READ-ONLY CHECK COMPLETED
No project files were modified by this cell.


In [18]:

from pathlib import Path
from google.colab import files
import ast
import hashlib

app_path = Path("/content/sketchforge3d/gradio_app.py")

if not app_path.is_file():
    print("❌ Gradio application file not found.")
else:
    source = app_path.read_text(encoding="utf-8")

    print("=== GRADIO SOURCE INSPECTION ===")
    print("File:", app_path.name)
    print("Lines:", len(source.splitlines()))
    print("SHA256:", hashlib.sha256(source.encode()).hexdigest())

    try:
        ast.parse(source)
        print("✅ Python syntax: VALID")
    except SyntaxError as error:
        print("❌ Python syntax error:", error)

    print("\nDownloading source for detailed review...")
    files.download(str(app_path))


❌ Gradio application file not found.


In [19]:

import gradio as gr
import inspect

print("Gradio version:", gr.__version__)

image_args = inspect.signature(
    gr.Image.__init__
).parameters

editor_args = inspect.signature(
    gr.ImageEditor.__init__
).parameters

print("\nComponent compatibility:")
print("Image supports old 'source':", "source" in image_args)
print("Image supports old 'tool':", "tool" in image_args)
print("ImageEditor available:", hasattr(gr, "ImageEditor"))
print("ImageEditor supports 'type':", "type" in editor_args)
print("Model3D available:", hasattr(gr, "Model3D"))
print("Sketchpad available:", hasattr(gr, "Sketchpad"))

print("\n✅ Read-only compatibility check completed.")


Gradio version: 6.30.0

Component compatibility:
Image supports old 'source': False
Image supports old 'tool': False
ImageEditor available: True
ImageEditor supports 'type': True
Model3D available: True
Sketchpad available: True

✅ Read-only compatibility check completed.


In [20]:

import os
import json
import uuid
import subprocess
from pathlib import Path
from PIL import Image
import gradio as gr

SAFE_ROOT = Path("/content/sketchforge3d")
SAFE_PYTHON = SAFE_ROOT / ".venv" / "bin" / "python"

assert (SAFE_ROOT / "engine.py").is_file(), "Original engine missing"
assert SAFE_PYTHON.is_file(), "Original environment missing"

# Preserve the existing inference environment.
SAFE_ENV = dict(globals().get("RUN_ENV", os.environ))
SAFE_ENV.pop("PYTHONPATH", None)
SAFE_ENV.update({
    "HF_HOME": str(SAFE_ROOT / "cache" / "huggingface"),
    "U2NET_HOME": str(SAFE_ROOT / "cache" / "rembg"),
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONUNBUFFERED": "1",
})

def safe_generate(
    source_mode, uploaded, drawing, prompt,
    quality, seed, strength, candidates,
    bake_texture, strict_texture, export_obj
):
    if strict_texture and not bake_texture:
        raise gr.Error("Enable UV baking before strict mode.")

    # Select the requested input.
    if source_mode == "Upload":
        image = uploaded
    else:
        image = None
        if isinstance(drawing, dict):
            image = drawing.get("composite")
            if image is None:
                image = drawing.get("background")

    if image is None:
        raise gr.Error("Please upload or draw a sketch.")

    # Composite transparent pixels over white.
    if isinstance(image, str):
        image = Image.open(image)

    if image.width * image.height > 25_000_000:
        raise gr.Error("Please use an image under 25 million pixels.")

    rgba = image.convert("RGBA")
    white = Image.new("RGBA", rgba.size, "white")
    white.alpha_composite(rgba)
    prepared = white.convert("RGB")

    # New locations only. Existing runs are untouched.
    run_id = uuid.uuid4().hex

    input_dir = SAFE_ROOT / "gradio_safe_inputs"
    input_dir.mkdir(parents=True, exist_ok=True)

    input_path = input_dir / f"{run_id}.png"
    prepared.save(input_path)

    output_dir = SAFE_ROOT / "gradio_safe_runs" / run_id

    command = [
        str(SAFE_PYTHON),
        str(SAFE_ROOT / "engine.py"),
        "run",
        "--input", str(input_path),
        "--run-dir", str(output_dir),
        "--triposr-dir", str(SAFE_ROOT / "TripoSR"),
        "--prompt", str(prompt or ""),
        "--quality", str(quality),
        "--seed", str(int(seed)),
        "--control-strength", str(float(strength)),
        "--candidates", str(int(candidates)),
    ]

    if bake_texture:
        command.append("--texture")
    if strict_texture:
        command.append("--require-texture")
    if export_obj:
        command.append("--obj")

    result = subprocess.run(
        command,
        cwd=str(SAFE_ROOT),
        env=SAFE_ENV,
        capture_output=True,
        text=True,
    )

    report_path = output_dir / "report.json"

    report = {}
    if report_path.is_file():
        report = json.loads(report_path.read_text())

    success = (
        result.returncode == 0
        and report.get("status") == "success"
        and (output_dir / "model.glb").is_file()
    )

    images = [
        ("00_input.png", "Input sketch"),
        ("02_control.png", "Sketch guidance"),
        ("03_product.png", "Generated image"),
        ("04_foreground.png", "Foreground"),
        ("05_triposr_input.png", "TripoSR input"),
    ]

    gallery = [
        (str(output_dir / filename), label)
        for filename, label in images
        if (output_dir / filename).is_file()
    ]

    model = str(output_dir / "model.glb") if success else None

    bundle = output_dir / "result_bundle.zip"
    download = str(bundle) if bundle.is_file() else None

    if success:
        appearance = report.get("reconstruction", {}).get(
            "appearance", "unknown"
        )
        time_taken = report.get("total_time_s", "unknown")

        status = (
            f"### ✅ Generation successful\n"
            f"**Time:** {time_taken} seconds\n\n"
            f"**Appearance:** {appearance}\n\n"
            f"**Backend:** TripoSR"
        )
    else:
        error = report.get(
            "error",
            f"Process exited with code {result.returncode}"
        )
        status = f"### ❌ Generation failed\n{error}"

    logs = (result.stdout or "") + (result.stderr or "")

    return (
        status,
        model,
        gallery,
        download,
        report,
        logs[-10000:]
    )


with gr.Blocks(
    title="SketchForge 3D — Safe Interface",
    theme=gr.themes.Soft()
) as safe_demo:

    gr.Markdown(
        "# SketchForge 3D\n"
        "### 2D Sketch → 3D Asset\n"
        "Stable Diffusion + ControlNet + TripoSR"
    )

    with gr.Row():

        with gr.Column(scale=1):

            gr.Markdown("### Step 1 — Sketch Input")

            source_mode = gr.Radio(
                ["Upload", "Draw"],
                value="Upload",
                label="Input mode"
            )

            uploaded = gr.Image(
                label="Upload your sketch",
                sources=["upload"],
                type="pil",
                height=280
            )

            drawing = gr.ImageEditor(
                label="Draw your sketch (use black strokes)",
                value=Image.new("RGB", (512, 512), "white"),
                type="pil",
                height=320
            )

            prompt = gr.Textbox(
                label="Optional description",
                placeholder="e.g. a ceramic mug with a handle"
            )

            quality = gr.Dropdown(
                choices=["preview", "balanced", "detailed"],
                value="preview",
                label="Quality"
            )

            with gr.Accordion("Advanced Settings", open=True):

                seed = gr.Number(
                    value=42, precision=0, label="Seed"
                )

                strength = gr.Slider(
                    0.5, 1.5, value=1.1, step=0.1,
                    label="Sketch guidance"
                )

                candidates = gr.Slider(
                    1, 2, value=1, step=1,
                    label="Candidates"
                )

                bake = gr.Checkbox(
                    label="Bake UV texture"
                )

                strict = gr.Checkbox(
                    label="Strict UV texture mode"
                )

                export_obj = gr.Checkbox(
                    label="Export OBJ"
                )

            generate = gr.Button(
                "Generate 3D Model",
                variant="primary"
            )

        with gr.Column(scale=1):

            gr.Markdown("### Step 2 — Generated 3D Asset")

            status = gr.Markdown("Ready for generation.")

            viewer = gr.Model3D(
                label="Interactive 3D Viewer",
                height=420
            )

            download = gr.File(
                label="Download result ZIP"
            )

    gr.Markdown("### Pipeline Processing Stages")

    gallery = gr.Gallery(
        label="Intermediate Results",
        columns=3
    )

    with gr.Accordion("Execution Details", open=False):
        report_output = gr.JSON(label="Validation Report")
        log_output = gr.Textbox(label="Pipeline Logs", lines=12)

    generate.click(
        fn=safe_generate,
        inputs=[
            source_mode, uploaded, drawing, prompt,
            quality, seed, strength, candidates,
            bake, strict, export_obj
        ],
        outputs=[
            status, viewer, gallery, download,
            report_output, log_output
        ]
    )

safe_demo.queue(default_concurrency_limit=1)

print("✅ Safe Gradio interface prepared.")
print("✅ Existing project files remain untouched.")
print("✅ No server has been launched.")
print("✅ No model inference has started.")


/tmp/ipykernel_4370/3606933000.py:160: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ Safe Gradio interface prepared.
✅ Existing project files remain untouched.
✅ No server has been launched.
✅ No model inference has started.


In [21]:

from pathlib import Path
import subprocess

PROJECT_ROOT = Path("/content/sketchforge3d")
VENV_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"

print("=== CURRENT ISOLATED ENVIRONMENT ===")

check_code = """
import importlib.metadata as m

packages = [
    "huggingface-hub",
    "transformers",
    "diffusers",
    "accelerate",
    "gradio",
    "torch",
    "torchvision"
]

for package in packages:
    try:
        print(f"{package}: {m.version(package)}")
    except m.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")
"""

result = subprocess.run(
    [str(VENV_PYTHON), "-c", check_code],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("Diagnostic errors:", result.stderr)

print("\n=== ORIGINAL LOCKFILE VERSIONS ===")

lockfile = PROJECT_ROOT / "requirements.lock"

for line in lockfile.read_text().splitlines():
    if line.startswith((
        "huggingface-hub==",
        "transformers==",
        "diffusers==",
        "accelerate==",
        "torch==",
        "torchvision=="
    )):
        print(line)

print("\n✅ Inspection complete. No packages changed.")


=== CURRENT ISOLATED ENVIRONMENT ===
huggingface-hub: 0.25.2
transformers: 4.44.2
diffusers: 0.30.3
accelerate: 0.34.2
gradio: NOT INSTALLED
torch: 2.6.0
torchvision: 0.21.0


=== ORIGINAL LOCKFILE VERSIONS ===
accelerate==0.34.2
diffusers==0.30.3
huggingface-hub==0.25.2
torch==2.6.0
torchvision==0.21.0
transformers==4.44.2

✅ Inspection complete. No packages changed.


In [22]:

from pathlib import Path
import subprocess
import sys
import os

PROJECT_ROOT = Path("/content/sketchforge3d")
VENV_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"
LOCKFILE = PROJECT_ROOT / "requirements.lock"
UV_TOOLS = PROJECT_ROOT / ".bootstrap"

assert VENV_PYTHON.is_file(), "Inference environment not found"
assert LOCKFILE.is_file(), "Original lockfile not found"
assert UV_TOOLS.is_dir(), "Original uv bootstrap tools not found"

print("=" * 50)
print("RESTORING ORIGINAL SKETCHFORGE ML ENVIRONMENT")
print("=" * 50)

# Use the existing uv installation from the original bootstrap.
repair_env = dict(os.environ)
repair_env["PYTHONPATH"] = str(UV_TOOLS)
repair_env["UV_CACHE_DIR"] = str(PROJECT_ROOT / ".uv-cache")

# Restore precisely the packages recorded in requirements.lock.
subprocess.run(
    [
        sys.executable,
        "-m", "uv",
        "pip", "sync",
        "--python", str(VENV_PYTHON),
        str(LOCKFILE)
    ],
    cwd=str(PROJECT_ROOT),
    env=repair_env,
    check=True
)

# Check dependency consistency.
subprocess.run(
    [
        sys.executable,
        "-m", "uv",
        "pip", "check",
        "--python", str(VENV_PYTHON)
    ],
    cwd=str(PROJECT_ROOT),
    env=repair_env,
    check=True
)

# Verify critical versions.
verify_code = """
import importlib.metadata as m

for name in [
    "huggingface-hub",
    "transformers",
    "diffusers",
    "torch"
]:
    print(f"{name}: {m.version(name)}")
"""

subprocess.run(
    [str(VENV_PYTHON), "-c", verify_code],
    check=True
)

print("\n✅ Original ML dependency environment restored.")
print("✅ Source code and previous results remain untouched.")


RESTORING ORIGINAL SKETCHFORGE ML ENVIRONMENT

✅ Original ML dependency environment restored.
✅ Source code and previous results remain untouched.


In [23]:

from pathlib import Path
import subprocess

PROJECT_ROOT = Path("/content/sketchforge3d")
VENV_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"

test_code = """
import torch
import transformers
import diffusers
import huggingface_hub

from diffusers import (
    StableDiffusionControlNetPipeline,
    ControlNetModel
)
from transformers import CLIPTextModel

print("✅ PyTorch:", torch.__version__)
print("✅ Transformers:", transformers.__version__)
print("✅ Diffusers:", diffusers.__version__)
print("✅ Hugging Face Hub:", huggingface_hub.__version__)
print("✅ CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))

print("✅ Core ML imports successful")
"""

result = subprocess.run(
    [str(VENV_PYTHON), "-c", test_code],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("❌ Backend test failed:")
    print(result.stderr)
else:
    print("✅ BACKEND COMPATIBILITY TEST PASSED")


✅ PyTorch: 2.6.0+cu124
✅ Transformers: 4.44.2
✅ Diffusers: 0.30.3
✅ Hugging Face Hub: 0.25.2
✅ CUDA available: True
✅ GPU: Tesla T4
✅ Core ML imports successful

✅ BACKEND COMPATIBILITY TEST PASSED


In [24]:

from pathlib import Path
import json
import struct
import hashlib

ROOT = Path("/content/sketchforge3d")

# Find the latest successful strict UV run.
candidates = []

for folder in ["gradio_safe_runs", "runs_gradio", "runs"]:
    base = ROOT / folder

    if not base.is_dir():
        continue

    for run in base.iterdir():
        if not run.is_dir():
            continue

        config_file = run / "config.json"
        report_file = run / "report.json"
        model_file = run / "model.glb"

        if not all(p.is_file() for p in [
            config_file, report_file, model_file
        ]):
            continue

        config = json.loads(config_file.read_text())
        report = json.loads(report_file.read_text())

        if (
            config.get("bake_texture") is True
            and config.get("require_texture") is True
            and report.get("status") == "success"
        ):
            candidates.append(model_file)

assert candidates, "No successful strict UV model found."

original = max(
    candidates,
    key=lambda p: p.stat().st_mtime
)

print("Original model:", original)

data = original.read_bytes()
original_hash = hashlib.sha256(data).hexdigest()

# Read GLB header and JSON chunk.
magic, version, total = struct.unpack_from(
    "<4sII", data, 0
)

assert magic == b"glTF"
assert version == 2
assert total == len(data)

json_length, chunk_type = struct.unpack_from(
    "<I4s", data, 12
)

assert chunk_type == b"JSON"

metadata = json.loads(
    data[20:20 + json_length]
)

# Correct only the erroneous material multiplier.
changed = 0

for material in metadata.get("materials", []):
    pbr = material.get("pbrMetallicRoughness", {})

    if "baseColorTexture" not in pbr:
        continue

    factor = pbr.get("baseColorFactor")

    if factor is None:
        continue

    if all(abs(x - 1 / 255) < 1e-6 for x in factor):
        pbr["baseColorFactor"] = [1.0, 1.0, 1.0, 1.0]
        changed += 1

assert changed > 0, (
    "The expected material issue was not found. "
    "No corrected model was created."
)

# Rebuild the JSON chunk while preserving the binary data.
new_json = json.dumps(
    metadata,
    separators=(",", ":")
).encode("utf-8")

new_json += b" " * ((-len(new_json)) % 4)

remaining_chunks = data[20 + json_length:]

new_chunk = (
    struct.pack("<I4s", len(new_json), b"JSON")
    + new_json
)

new_length = 12 + len(new_chunk) + len(remaining_chunks)

corrected_data = (
    struct.pack("<4sII", b"glTF", 2, new_length)
    + new_chunk
    + remaining_chunks
)

# Save to a NEW location, without overwriting anything.
output_dir = ROOT / "uv_color_preview_copies" / original.parent.name
output_dir.mkdir(parents=True, exist_ok=True)

corrected = output_dir / "model_uv_color_fixed.glb"

with corrected.open("xb") as stream:
    stream.write(corrected_data)

# Verify the new file.
saved = corrected.read_bytes()

assert len(saved) == new_length
assert saved[20 + len(new_json):] != b""

new_json_length = struct.unpack_from("<I", saved, 12)[0]

new_metadata = json.loads(
    saved[20:20 + new_json_length]
)

new_factor = new_metadata["materials"][0][
    "pbrMetallicRoughness"
]["baseColorFactor"]

assert new_factor == [1.0, 1.0, 1.0, 1.0]

# Confirm original is unchanged.
assert hashlib.sha256(
    original.read_bytes()
).hexdigest() == original_hash

print("\n✅ Corrected GLB created successfully")
print("✅ New material factor:", new_factor)
print("✅ Original GLB unchanged")
print("✅ Original source code unchanged")
print("✅ No model inference performed")
print("\nCorrected file:", corrected)


AssertionError: No successful strict UV model found.

In [25]:

from pathlib import Path
import hashlib
import difflib

ROOT = Path("/content/sketchforge3d")

original = ROOT / "engine.py"
corrected = ROOT / "engine_uv_fixed.py"

assert original.is_file(), "Original engine not found"

# Never overwrite an existing corrected copy.
assert not corrected.exists(), (
    "engine_uv_fixed.py already exists. "
    "Stop rather than overwrite it."
)

original_code = original.read_text(encoding="utf-8")
original_hash = hashlib.sha256(
    original.read_bytes()
).hexdigest()

old = "baseColorFactor=[1, 1, 1, 1]"
new = "baseColorFactor=[255, 255, 255, 255]"

# Ensure exactly one intended occurrence.
assert original_code.count(old) == 1, (
    "Expected material-setting line not found exactly once."
)

corrected_code = original_code.replace(old, new, 1)

# Validate Python syntax before creating the file.
compile(corrected_code, str(corrected), "exec")

# Create a NEW file only.
with corrected.open("x", encoding="utf-8") as f:
    f.write(corrected_code)

# Verify the original was not modified.
assert hashlib.sha256(
    original.read_bytes()
).hexdigest() == original_hash

# Confirm only the intended replacement occurred.
assert corrected.read_text(encoding="utf-8") == (
    original_code.replace(old, new, 1)
)

print("=== BACKEND COPY VERIFICATION ===")

print("✅ Corrected backend created")
print("✅ Python syntax valid")
print("✅ Exactly one material-setting change")
print("✅ Original engine.py unchanged")
print("✅ No dependencies modified")
print("✅ No GPU inference performed")

print("\nCorrected backend:", corrected)

print("\n=== EXACT CODE DIFFERENCE ===")

difference = difflib.unified_diff(
    original_code.splitlines(),
    corrected_code.splitlines(),
    fromfile="engine.py",
    tofile="engine_uv_fixed.py",
    lineterm=""
)

for line in difference:
    print(line)


=== BACKEND COPY VERIFICATION ===
✅ Corrected backend created
✅ Python syntax valid
✅ Exactly one material-setting change
✅ Original engine.py unchanged
✅ No dependencies modified
✅ No GPU inference performed

Corrected backend: /content/sketchforge3d/engine_uv_fixed.py

=== EXACT CODE DIFFERENCE ===
--- engine.py
+++ engine_uv_fixed.py
@@ -315,7 +315,7 @@
     nearest = distance_transform_edt(~covered, return_distances=False, return_indices=True)
     texture[~covered] = texture[nearest[0][~covered], nearest[1][~covered]]
     texture_image = Image.fromarray(texture, "RGB")
-    material = trimesh.visual.material.PBRMaterial(baseColorTexture=texture_image, baseColorFactor=[1, 1, 1, 1], metallicFactor=0.0, roughnessFactor=0.75)
+    material = trimesh.visual.material.PBRMaterial(baseColorTexture=texture_image, baseColorFactor=[255, 255, 255, 255], metallicFactor=0.0, roughnessFactor=0.75)
     output = trimesh.Trimesh(vertices=verts, faces=indices, process=False)
     output.visual = t

In [26]:

import secrets

# Generate temporary login credentials
SAFE_USERNAME = "team"
SAFE_PASSWORD = secrets.token_urlsafe(16)

print("=" * 45)
print("SKETCHFORGE 3D — SECURE GRADIO LAUNCH")
print("=" * 45)

print("\nUsername:", SAFE_USERNAME)
print("Password:", SAFE_PASSWORD)

print("\nKeep these credentials private.")
print("Starting Gradio interface...\n")

safe_demo.launch(
    share=True,
    auth=(SAFE_USERNAME, SAFE_PASSWORD),
    inline=True,
    show_error=True,
    theme=gr.themes.Soft()
)


SKETCHFORGE 3D — SECURE GRADIO LAUNCH

Username: team
Password: 5ezN5ImVTeMFCHfdsE1oug

Keep these credentials private.
Starting Gradio interface...

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://66dfc76d024c23d56b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


MAJOR TESTING(MUG) and handling with diagnosis

Diagnose the unwanted oval artifact

In [27]:

from pathlib import Path
from PIL import Image
from IPython.display import display
import numpy as np
import json

ROOT = Path("/content/sketchforge3d")

print("=" * 55)
print("SKETCHFORGE 3D — READ-ONLY FOREGROUND DIAGNOSTIC")
print("=" * 55)

# Locate the specific short-prompt test.
matching_runs = []

for report_path in ROOT.rglob("report.json"):
    try:
        report = json.loads(report_path.read_text())
        config = json.loads(
            (report_path.parent / "config.json").read_text()
        )

        prompt = config.get("prompt", "")

        if (
            "Photorealistic plain white ceramic coffee mug"
            in prompt
        ):
            matching_runs.append(report_path.parent)

    except (OSError, ValueError, KeyError):
        continue

if not matching_runs:
    print("❌ Short-prompt test not found.")
    print("No files have been changed.")
else:
    run_dir = max(
        matching_runs,
        key=lambda p: (p / "report.json").stat().st_mtime
    )

    print("\n✅ Found test:", run_dir)

    image_files = [
        ("Generated image", "03_product.png"),
        ("Foreground mask", "04_mask.png"),
        ("Isolated foreground", "04_foreground.png"),
        ("TripoSR input", "05_triposr_input.png"),
    ]

    for title, filename in image_files:
        path = run_dir / filename

        print("\n" + title)

        if path.exists():
            display(Image.open(path))
        else:
            print("⚠️ Missing:", filename)

    mask_path = run_dir / "04_mask.png"

    if mask_path.exists():
        mask = np.asarray(
            Image.open(mask_path).convert("L")
        )

        binary = mask > 127

        print("\n=== FOREGROUND MASK STATISTICS ===")
        print("Mask size:", mask.shape)
        print("Foreground coverage:",
              round(binary.mean() * 100, 2), "%")

        try:
            from scipy.ndimage import label

            components, count = label(binary)

            sizes = np.bincount(
                components[binary].ravel(),
                minlength=count + 1
            )[1:]

            print("Disconnected mask regions:", count)
            print(
                "Largest region sizes:",
                sorted(sizes.tolist(), reverse=True)[:10]
            )

        except ImportError:
            print(
                "SciPy unavailable; component count skipped."
            )

print("\n✅ Diagnostic finished.")
print("✅ No source files modified.")
print("✅ No inference performed.")


SKETCHFORGE 3D — READ-ONLY FOREGROUND DIAGNOSTIC
❌ Short-prompt test not found.
No files have been changed.

✅ Diagnostic finished.
✅ No source files modified.
✅ No inference performed.


Preview the foreground-mask correction.

In [28]:

from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
import numpy as np
import json

ROOT = Path("/content/sketchforge3d")

print("=" * 55)
print("SKETCHFORGE 3D — SAFE MASK CORRECTION PREVIEW")
print("=" * 55)

# Find the previous short-prompt mug test.
matching_runs = []

for config_path in ROOT.rglob("config.json"):
    try:
        config = json.loads(config_path.read_text())

        if "Photorealistic plain white ceramic coffee mug" in config.get("prompt", ""):
            mask_path = config_path.parent / "04_mask.png"

            if mask_path.exists():
                matching_runs.append(mask_path)

    except (OSError, ValueError):
        continue

if not matching_runs:
    print("❌ Could not find the short-prompt mug mask.")
    print("No changes made.")

else:
    mask_path = max(
        matching_runs,
        key=lambda p: p.stat().st_mtime
    )

    print("✅ Found:", mask_path)

    original_mask = Image.open(mask_path).convert("L")

    # Work entirely in memory.
    corrected_mask = original_mask.copy()

    # Coordinates are based on the inspected 512×512 image.
    if corrected_mask.size != (512, 512):
        print("⚠️ Unexpected mask dimensions. Correction skipped.")

    else:
        # Experimental polygon covering the unwanted lower-right oval.
        # This is ONLY a preview; the handle must be inspected.
        removal_polygon = [
            (338, 377),
            (365, 361),
            (417, 340),
            (460, 348),
            (484, 366),
            (492, 397),
            (468, 422),
            (420, 433),
            (353, 431),
            (337, 412),
        ]

        draw = ImageDraw.Draw(corrected_mask)
        draw.polygon(removal_polygon, fill=0)

        original = np.asarray(original_mask)
        corrected = np.asarray(corrected_mask)

        removed = (original > 127) & (corrected <= 127)

        print("\n=== PREVIEW STATISTICS ===")
        print("Original foreground pixels:",
              int((original > 127).sum()))

        print("Corrected foreground pixels:",
              int((corrected > 127).sum()))

        print("Removed foreground pixels:",
              int(removed.sum()))

        print("\n1. ORIGINAL MASK")
        display(original_mask)

        print("\n2. EXPERIMENTAL CORRECTED MASK")
        display(corrected_mask)

        print("\n3. REMOVED AREA (RED)")
        overlay = Image.fromarray(
            np.stack([original] * 3, axis=-1)
        )

        overlay_array = np.array(overlay)
        overlay_array[removed] = [255, 0, 0]

        display(Image.fromarray(overlay_array))

        print("\n✅ Preview completed.")
        print("✅ No files saved or modified.")
        print("✅ No inference performed.")


SKETCHFORGE 3D — SAFE MASK CORRECTION PREVIEW
❌ Could not find the short-prompt mug mask.
No changes made.


## What to check before submission

- Run a fresh sketch through all stages with the TripoSR backend.
- Confirm the optional-description path works with an empty prompt.
- Reopen the GLB in Blender and inspect geometry, thin parts, colors/textures, side and back.
- When claiming texture generation, require and inspect a UV texture; vertex colors are reported separately.
- Repeat several runs and inspect per-stage GPU memory and timing reports.
- Start a fresh runtime and follow this notebook from the beginning.
- Submit code, environment information, model/license attribution, sample inputs and actual generated assets.

First-run download time is part of a cold run. Reports do not claim a fixed generation speed. Free Colab resource availability varies, and local `/content` files can disappear when a session is replaced.

## Troubleshooting

**No GPU:** select a GPU runtime. **Memory error:** start with Preview, one candidate, no texture, and stop other GPU work. **Wrong shape:** check the control map, object image and foreground mask before changing mesh resolution. **Background isolation failed:** use one complete object and a plain background. **Texture failed:** inspect the warning; a non-strict run can still export vertex colors. **Setup failed:** rerun the setup cell after checking its exact error; keep the locked versions.

## Architecture, model pins and licensing

Source files are embedded above. `engine.py` contains the pipeline, CLI, immutable model revisions and adapters. `requirements.lock` pins the resolved Linux package versions. The setup records the installed environment and checks out the exact TripoSR code commit.

- [Stable Diffusion 1.5](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5): CreativeML OpenRAIL-M, with usage restrictions.
- [ControlNet Scribble](https://huggingface.co/lllyasviel/control_v11p_sd15_scribble): see the model card and OpenRAIL terms.
- [TripoSR code](https://github.com/VAST-AI-Research/TripoSR) and [weights](https://huggingface.co/stabilityai/TripoSR): MIT.
- [rembg](https://github.com/danielgatis/rembg): MIT library; review the IS-Net model/project terms separately.
- [PyMCubes](https://github.com/pmneila/PyMCubes), [xatlas](https://github.com/mworchel/xatlas-python), [trimesh](https://github.com/mikedh/trimesh), [model-viewer](https://modelviewer.dev/), PyTorch, Diffusers and Transformers retain their respective licenses.

The model safety checker remains enabled. No paid generation service is called. Full reproduction and CLI instructions are embedded in `README.md`. [Colab resource and interface policies](https://research.google.com/colaboratory/faq.html).

In [30]:

from pathlib import Path

ROOT = Path("/content/sketchforge3d")

print("=== SKETCHFORGE 3D — GITHUB PREPARATION ===")

for name in [
    "engine.py",
    "engine_uv_fixed.py",
    "bootstrap.py",
    "notebook_app.py",
    "gradio_app.py",
    "test_engine.py",
    "requirements.lock",
    "README.md",
    "evaluation.csv",
]:
    path = ROOT / name
    print(("✅" if path.exists() else "⚠️"), name)

print("\n=== SAVED 3D RESULTS ===")

models = sorted(ROOT.rglob("model.glb"))

for model in models:
    if ".venv" not in model.parts:
        print(
            model.relative_to(ROOT),
            "|",
            round(model.stat().st_size / 1024 / 1024, 2),
            "MB"
        )

print("\n=== PROJECT SIZE ===")

for folder in [
    "runs",
    "gradio_safe_runs",
    "uv_color_preview_copies",
]:
    path = ROOT / folder

    if path.exists():
        files = [p for p in path.rglob("*") if p.is_file()]
        size_mb = sum(p.stat().st_size for p in files) / 1024**2

        print(folder, ":", round(size_mb, 2), "MB")

print("\n✅ Inspection complete. Nothing modified.")


=== SKETCHFORGE 3D — GITHUB PREPARATION ===
✅ engine.py
✅ engine_uv_fixed.py
✅ bootstrap.py
✅ notebook_app.py
⚠️ gradio_app.py
✅ test_engine.py
✅ requirements.lock
✅ README.md
✅ evaluation.csv

=== SAVED 3D RESULTS ===
runs/20261101_003249_49f2f14a/model.glb | 0.82 MB
runs/20261101_003401_e6434789/model.glb | 0.79 MB

=== PROJECT SIZE ===
runs : 7.89 MB

✅ Inspection complete. Nothing modified.


In [31]:
from pathlib import Path
import shutil
import re

ROOT = Path("/content/sketchforge3d")
STAGE = Path("/content/SketchForge3D_GitHub")

print("=" * 55)
print("SKETCHFORGE 3D — SAFE GITHUB STAGING")
print("=" * 55)

if not ROOT.is_dir():
    raise RuntimeError("Original project directory not found.")

if STAGE.exists():
    raise RuntimeError(
        "Staging folder already exists. "
        "No files changed. Inspect it before proceeding."
    )

# Explicitly select files for public GitHub.
source_files = [
    "engine.py",
    "engine_uv_fixed.py",
    "bootstrap.py",
    "notebook_app.py",
    "gradio_app.py",
    "test_engine.py",
    "README.md",
    "requirements.lock",
    "requirements.txt",
    "evaluation.csv",
    "verification.json",
]

selected = []

for name in source_files:
    path = ROOT / name
    if path.is_file():
        selected.append((path, Path(name)))

# Include generated evidence, not downloaded model weights.
allowed_result_files = {
    "model.glb",
    "texture.png",
    "00_input.png",
    "03_product.png",
    "04_foreground.png",
    "04_mask.png",
    "05_triposr_input.png",
    "config.json",
    "report.json",
    "reconstruction.json",
}

runs_dir = ROOT / "runs"

if runs_dir.exists():
    for run in sorted(runs_dir.iterdir()):
        if not run.is_dir():
            continue

        for name in allowed_result_files:
            path = run / name

            if path.is_file():
                selected.append(
                    (path, Path("runs") / run.name / name)
                )

# Check selected text files for common credential indicators.
secret_patterns = [
    r"ghp_[A-Za-z0-9]{20,}",
    r"github_pat_[A-Za-z0-9_]{20,}",
    r"hf_[A-Za-z0-9]{20,}",
    r"AIza[0-9A-Za-z_-]{20,}",
    r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----",
    r"sk-[A-Za-z0-9]{20,}",
]

suspect_files = []

for source, destination in selected:
    if source.suffix.lower() not in {
        ".py", ".json", ".md", ".txt", ".csv", ".lock"
    }:
        continue

    content = source.read_text(
        encoding="utf-8",
        errors="replace"
    )

    if any(
        re.search(pattern, content)
        for pattern in secret_patterns
    ):
        suspect_files.append(str(destination))

if suspect_files:
    print("\n⚠️ POSSIBLE SECRETS DETECTED")
    for filename in suspect_files:
        print("Review:", filename)

    raise RuntimeError(
        "Public upload preparation stopped. "
        "Review flagged files first."
    )

# Create a new upload copy.
STAGE.mkdir(parents=True)

for source, destination in selected:
    target = STAGE / destination
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)

print("\n✅ Files copied:", len(selected))

print("\n=== STAGED TOP-LEVEL FILES ===")
for path in sorted(STAGE.iterdir()):
    print("📁" if path.is_dir() else "📄", path.name)

print("\n=== RESULT MODELS ===")
for model in sorted(STAGE.rglob("model.glb")):
    print(model.relative_to(STAGE))

print("\n=== NOTEBOOK CHECK ===")
notebooks = [
    p for p in ROOT.glob("*.ipynb")
    if p.is_file()
]

if notebooks:
    print("Notebook candidates found:")
    for notebook in notebooks:
        print(" -", notebook.name)
    print("⚠️ Notebook not copied yet; review it first.")
else:
    print("⚠️ Notebook not found in project directory.")
    print("We'll export it directly from Colab.")

print("\n✅ GitHub staging prepared.")
print("✅ Original project files unchanged.")
print("✅ No GitHub upload performed.")

SKETCHFORGE 3D — SAFE GITHUB STAGING

✅ Files copied: 27

=== STAGED TOP-LEVEL FILES ===
📄 README.md
📄 bootstrap.py
📄 engine.py
📄 engine_uv_fixed.py
📄 evaluation.csv
📄 notebook_app.py
📄 requirements.lock
📁 runs
📄 test_engine.py
📄 verification.json

=== RESULT MODELS ===
runs/20261101_003249_49f2f14a/model.glb
runs/20261101_003401_e6434789/model.glb

=== NOTEBOOK CHECK ===
⚠️ Notebook not found in project directory.
We'll export it directly from Colab.

✅ GitHub staging prepared.
✅ Original project files unchanged.
✅ No GitHub upload performed.
